# 03 — Train/Test Split et Preprocessing Machine Learning

## Objectif

Ce notebook prépare le dataset issu du Feature Engineering pour l'entraînement
des modèles de régression.

Les objectifs sont :

- charger le dataset de features ;
- séparer les variables explicatives `X` et la cible `y` ;
- créer les jeux d'entraînement et de test ;
- identifier les variables numériques et catégorielles ;
- définir les stratégies d'imputation ;
- définir les stratégies d'encodage des variables catégorielles ;
- construire et appliquer les différentes étapes du preprocessing Machine Learning ;
- ajuster le preprocessing uniquement sur le jeu d'entraînement ;
- transformer les jeux d'entraînement et de test ;
- vérifier la cohérence des matrices obtenues.

La variable cible est :

`co2_wltp_g_km`

Aucune transformation dépendant des données n'est ajustée avant la séparation
train / test.

In [1]:
from pathlib import Path

import pandas as pd

from sklearn.model_selection import train_test_split


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "03_ml_preprocessing":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

INPUT_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "data_2024_features.csv"
)

TARGET = "co2_wltp_g_km"

TEST_MODE = True
NROWS_TEST = 100_000

TEST_SIZE = 0.20
RANDOM_STATE = 42

## 1. Chargement du dataset de features

### Objectif

Cette étape charge le dataset produit par le pipeline de Feature Engineering.

Deux modes sont utilisés :

- **Mode TEST** : 100 000 observations pour le développement local ;
- **Mode COMPLET** : intégralité du dataset pour la validation finale.

Aucune transformation n'est appliquée lors du chargement.

In [2]:
if not INPUT_DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset de features introuvable : {INPUT_DATA_PATH}"
    )

if TEST_MODE:
    df = pd.read_csv(
        INPUT_DATA_PATH,
        nrows=NROWS_TEST,
        low_memory=False,
    )

    print(
        f"Mode TEST : {len(df):,} observations chargées."
    )
else:
    df = pd.read_csv(
        INPUT_DATA_PATH,
        low_memory=False,
    )

    print(
        f"Mode COMPLET : {len(df):,} observations chargées."
    )

print(f"Shape : {df.shape}")

Mode TEST : 100,000 observations chargées.
Shape : (100000, 15)


## 2. Contrôle du schéma d'entrée

### Objectif

Avant la séparation des données, cette étape vérifie :

- la présence de la variable cible ;
- l'absence de valeurs manquantes dans la cible ;
- l'unicité des noms de colonnes ;
- la cohérence générale du dataset de features.

In [3]:
if TARGET not in df.columns:
    raise ValueError(
        f"Variable cible absente : {TARGET}"
    )

if df[TARGET].isna().any():
    raise ValueError(
        "La variable cible contient des valeurs manquantes."
    )

if not df.columns.is_unique:
    raise ValueError(
        "Les noms de colonnes ne sont pas uniques."
    )

print("✅ Schéma d'entrée valide.")

✅ Schéma d'entrée valide.


## 3. Séparation entre variables explicatives et cible

### Objectif

Cette étape construit :

- `X` : ensemble des variables explicatives retenues pour la modélisation ;
- `y` : variable cible `co2_wltp_g_km`.

La variable cible `co2_wltp_g_km` est séparée des variables explicatives afin de
constituer `y`.

La variable `manufacturer_make` est, quant à elle, volontairement exclue des
variables explicatives retenues dans `X`.

Elle a été conservée dans les étapes précédentes afin de préserver l'information
disponible dans le dataset et de permettre son analyse.

Cependant, les analyses réalisées ont montré que cette variable présente :

- une forte diversité de modalités ;
- de nombreuses variantes de représentation d'une même marque ;
- un besoin important de normalisation métier ;
- des valeurs résiduelles difficiles à résoudre de manière fiable ;
- une information qui, prise isolément, n'apporte pas une justification suffisante
  pour être retenue comme variable prédictive.

Le traitement réalisé au Notebook 01 a permis d'améliorer la qualité de cette
variable, mais il a également confirmé sa complexité et son instabilité
sémantique.

Afin de conserver un ensemble de variables explicatives plus robuste et plus
facilement reproductible, `manufacturer_make` n'est donc pas retenue pour la
modélisation.

In [4]:
# ---------------------------------------------------------------------
# Variables exclues de la modélisation
# ---------------------------------------------------------------------

EXCLUDED_FEATURES = [
    "manufacturer_make",
]


# ---------------------------------------------------------------------
# Construction de X et y
# ---------------------------------------------------------------------

X = df.drop(
    columns=[
        TARGET,
        *EXCLUDED_FEATURES,
    ]
).copy()

y = df[TARGET].copy()


# ---------------------------------------------------------------------
# Contrôles de cohérence
# ---------------------------------------------------------------------

if TARGET in X.columns:
    raise ValueError(
        f"La variable cible '{TARGET}' est encore présente dans X."
    )

remaining_excluded_features = [
    column
    for column in EXCLUDED_FEATURES
    if column in X.columns
]

if remaining_excluded_features:
    raise ValueError(
        "Variables exclues encore présentes dans X : "
        + ", ".join(remaining_excluded_features)
    )


# ---------------------------------------------------------------------
# Résumé
# ---------------------------------------------------------------------

print(f"X : {X.shape}")
print(f"y : {y.shape}")

print("\nVariables exclues de la modélisation :")

for column in EXCLUDED_FEATURES:
    print(f"  - {column}")

X : (100000, 13)
y : (100000,)

Variables exclues de la modélisation :
  - manufacturer_make


## 4. Séparation Train / Test

### Objectif

Le dataset est séparé avant toute imputation, encodage ou standardisation.

Cette séparation garantit que les paramètres du preprocessing sont appris
uniquement à partir du jeu d'entraînement.

La répartition retenue est :

- 80 % pour l'entraînement ;
- 20 % pour le test ;
- `random_state = 42` pour assurer la reproductibilité.

In [5]:
# ---------------------------------------------------------------------
# Séparation Train / Test
# ---------------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)


# ---------------------------------------------------------------------
# Contrôles de cohérence
# ---------------------------------------------------------------------

if len(X_train) != len(y_train):
    raise ValueError(
        "Le nombre d'observations de X_train et y_train est différent."
    )

if len(X_test) != len(y_test):
    raise ValueError(
        "Le nombre d'observations de X_test et y_test est différent."
    )

if X_train.shape[1] != X_test.shape[1]:
    raise ValueError(
        "X_train et X_test ne possèdent pas le même nombre de variables."
    )


# ---------------------------------------------------------------------
# Résumé de la séparation
# ---------------------------------------------------------------------

print("=== Résultat de la séparation Train / Test ===")

print(
    f"\nJeu d'entraînement (X_train) : "
    f"{X_train.shape[0]:,} observations × "
    f"{X_train.shape[1]} variables explicatives"
)

print(
    f"Jeu de test (X_test)         : "
    f"{X_test.shape[0]:,} observations × "
    f"{X_test.shape[1]} variables explicatives"
)

print(
    f"\nCible d'entraînement (y_train) : "
    f"{y_train.shape[0]:,} valeurs"
)

print(
    f"Cible de test (y_test)          : "
    f"{y_test.shape[0]:,} valeurs"
)

print(
    "\nRépartition appliquée : "
    "80 % des observations pour l'entraînement "
    "et 20 % pour le test."
)

=== Résultat de la séparation Train / Test ===

Jeu d'entraînement (X_train) : 80,000 observations × 13 variables explicatives
Jeu de test (X_test)         : 20,000 observations × 13 variables explicatives

Cible d'entraînement (y_train) : 80,000 valeurs
Cible de test (y_test)          : 20,000 valeurs

Répartition appliquée : 80 % des observations pour l'entraînement et 20 % pour le test.


## 5. Identification des variables numériques et catégorielles

### Objectif

Le preprocessing dépend du type de variable.

Les variables sont séparées en :

- variables numériques ;
- variables catégorielles.

Cette distinction permettra de construire des pipelines de transformation
adaptés à chaque groupe.

In [6]:
# ---------------------------------------------------------------------
# Identification des variables numériques et catégorielles
# ---------------------------------------------------------------------

numeric_columns = (
    X_train
    .select_dtypes(include="number")
    .columns
    .tolist()
)

categorical_columns = (
    X_train
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)


# ---------------------------------------------------------------------
# Contrôles de cohérence
# ---------------------------------------------------------------------

identified_columns = (
    numeric_columns
    + categorical_columns
)

if len(identified_columns) != X_train.shape[1]:
    raise ValueError(
        "Le nombre de variables identifiées ne correspond pas "
        "au nombre de variables présentes dans X_train."
    )

if set(numeric_columns).intersection(categorical_columns):
    raise ValueError(
        "Certaines variables sont identifiées simultanément "
        "comme numériques et catégorielles."
    )


# ---------------------------------------------------------------------
# Résumé
# ---------------------------------------------------------------------

print(
    "=== Identification des variables numériques "
    "et catégorielles ==="
)

print(
    f"\nNombre total de variables explicatives analysées : "
    f"{X_train.shape[1]}"
)

print(
    f"\nVariables numériques identifiées : "
    f"{len(numeric_columns)}"
)

for column in numeric_columns:
    print(f"  - {column}")

print(
    f"\nVariables catégorielles identifiées : "
    f"{len(categorical_columns)}"
)

for column in categorical_columns:
    print(f"  - {column}")

print(
    "\nContrôle de cohérence : "
    f"{len(numeric_columns)} variables numériques + "
    f"{len(categorical_columns)} variables catégorielles = "
    f"{len(identified_columns)} variables explicatives."
)

=== Identification des variables numériques et catégorielles ===

Nombre total de variables explicatives analysées : 13

Variables numériques identifiées : 10
  - mass_running_order_kg
  - wltp_test_mass_kg
  - engine_capacity_cm3
  - engine_power_kw
  - electric_energy_consumption_wh_km
  - co2_reduction_wltp_g_km
  - fuel_consumption
  - electric_range_km
  - registration_month_sin
  - registration_month_cos

Variables catégorielles identifiées : 3
  - vehicle_category_type
  - fuel_type
  - fuel_mode

Contrôle de cohérence : 10 variables numériques + 3 variables catégorielles = 13 variables explicatives.


## 6. Analyse des valeurs manquantes après séparation Train / Test

### Objectif

Cette étape analyse les valeurs manquantes dans le jeu d'entraînement afin de
définir les stratégies d'imputation adaptées.

Les statistiques sont calculées uniquement sur `X_train`.

In [7]:
# ---------------------------------------------------------------------
# Analyse des valeurs manquantes dans X_train
# ---------------------------------------------------------------------

missing_summary = pd.DataFrame({
    "variable": X_train.columns,
    "type": X_train.dtypes.astype(str).values,
    "nombre_valeurs_manquantes": X_train.isna().sum().values,
    "pourcentage_valeurs_manquantes": (
        X_train.isna().mean().values * 100
    ),
})


# ---------------------------------------------------------------------
# Conservation uniquement des variables contenant des NaN
# ---------------------------------------------------------------------

missing_summary = (
    missing_summary[
        missing_summary["nombre_valeurs_manquantes"] > 0
    ]
    .sort_values(
        "pourcentage_valeurs_manquantes",
        ascending=False,
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# Résumé contextualisé
# ---------------------------------------------------------------------

print(
    "=== Analyse des valeurs manquantes "
    "dans le jeu d'entraînement ==="
)

print(
    f"\nNombre de variables explicatives analysées : "
    f"{X_train.shape[1]}"
)

print(
    f"Variables contenant au moins une valeur manquante : "
    f"{len(missing_summary)}"
)

print(
    "\nLes statistiques ci-dessous sont calculées "
    "uniquement sur X_train afin de ne pas utiliser "
    "d'information provenant du jeu de test."
)


# ---------------------------------------------------------------------
# Affichage du diagnostic
# ---------------------------------------------------------------------

display(missing_summary)

=== Analyse des valeurs manquantes dans le jeu d'entraînement ===

Nombre de variables explicatives analysées : 13
Variables contenant au moins une valeur manquante : 7

Les statistiques ci-dessous sont calculées uniquement sur X_train afin de ne pas utiliser d'information provenant du jeu de test.


,variable,type,nombre_valeurs_manquantes,pourcentage_valeurs_manquantes
0,electric_range_km,float64,63215,79.01875
1,electric_energy_consumption_wh_km,float64,63196,78.99500
2,co2_reduction_wltp_g_km,float64,35042,43.80250
3,fuel_consumption,float64,11677,14.59625
4,engine_capacity_cm3,float64,11355,14.19375
5,wltp_test_mass_kg,float64,225,0.28125
6,engine_power_kw,float64,1,0.00125


## 7. Analyse de la cardinalité des variables catégorielles du Train

### Objectif

Cette étape mesure la cardinalité réelle des variables catégorielles dans le
jeu d'entraînement.

Cette information servira à choisir les stratégies d'encodage sans utiliser
les données du jeu de test.

In [8]:
# ---------------------------------------------------------------------
# Analyse de la cardinalité des variables catégorielles
# ---------------------------------------------------------------------

categorical_cardinality = pd.DataFrame({
    "variable": categorical_columns,

    "nombre_modalites": [
        X_train[column].nunique(
            dropna=False
        )
        for column in categorical_columns
    ],

    "nombre_valeurs_manquantes": [
        X_train[column].isna().sum()
        for column in categorical_columns
    ],

    "pourcentage_valeurs_manquantes": [
        X_train[column].isna().mean() * 100
        for column in categorical_columns
    ],
})


# ---------------------------------------------------------------------
# Classement par cardinalité croissante
# ---------------------------------------------------------------------

categorical_cardinality = (
    categorical_cardinality
    .sort_values(
        "nombre_modalites",
        ascending=True,
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# Contrôles de cohérence
# ---------------------------------------------------------------------

if len(categorical_cardinality) != len(categorical_columns):
    raise ValueError(
        "Le nombre de variables analysées ne correspond pas "
        "au nombre de variables catégorielles identifiées."
    )


# ---------------------------------------------------------------------
# Résumé contextualisé
# ---------------------------------------------------------------------

print(
    "=== Analyse de la cardinalité des variables "
    "catégorielles du jeu d'entraînement ==="
)

print(
    f"\nNombre de variables catégorielles analysées : "
    f"{len(categorical_columns)}"
)

print(
    "\nLa cardinalité correspond au nombre de modalités "
    "distinctes observées dans chaque variable catégorielle."
)

print(
    "Les statistiques sont calculées uniquement sur X_train."
)


# ---------------------------------------------------------------------
# Affichage du diagnostic
# ---------------------------------------------------------------------

display(categorical_cardinality)

=== Analyse de la cardinalité des variables catégorielles du jeu d'entraînement ===

Nombre de variables catégorielles analysées : 3

La cardinalité correspond au nombre de modalités distinctes observées dans chaque variable catégorielle.
Les statistiques sont calculées uniquement sur X_train.


,variable,nombre_modalites,nombre_valeurs_manquantes,pourcentage_valeurs_manquantes
0,vehicle_category_type,2,0,0.0
1,fuel_mode,6,0,0.0
2,fuel_type,9,0,0.0


## 8. Définition des groupes de variables pour le preprocessing

### Objectif

Les variables sont regroupées selon leur type afin de définir les
transformations adaptées au pipeline de preprocessing.

Le jeu d'entraînement contient :

- 10 variables numériques ;
- 3 variables catégorielles :
  - `vehicle_category_type` ;
  - `fuel_type` ;
  - `fuel_mode`.

Les variables numériques seront traitées selon les règles d'imputation
définies à partir de `X_train`.

Les trois variables catégorielles ne présentent aucune valeur manquante dans
le jeu d'entraînement.

Elles seront donc conservées pour l'étape d'encodage, sans imputation
catégorielle préalable.

Toutes les transformations nécessitant un apprentissage de paramètres seront
ajustées exclusivement à partir de `X_train`.

In [9]:
# ---------------------------------------------------------------------
# Définition des groupes de variables pour le preprocessing
# ---------------------------------------------------------------------

categorical_columns_for_encoding = (
    categorical_columns.copy()
)


# ---------------------------------------------------------------------
# Contrôles de cohérence
# ---------------------------------------------------------------------

if set(categorical_columns_for_encoding) != set(
    [
        "vehicle_category_type",
        "fuel_type",
        "fuel_mode",
    ]
):
    raise ValueError(
        "Les variables catégorielles identifiées ne correspondent pas "
        "aux variables attendues pour l'encodage."
    )

if any(
    X_train[column].isna().any()
    for column in categorical_columns_for_encoding
):
    raise ValueError(
        "Au moins une variable catégorielle contient encore "
        "des valeurs manquantes dans X_train."
    )


# ---------------------------------------------------------------------
# Résumé contextualisé
# ---------------------------------------------------------------------

print(
    "=== Définition des groupes de variables "
    "pour le preprocessing ==="
)

print(
    f"\nVariables numériques : "
    f"{len(numeric_columns)}"
)

print(
    f"Variables catégorielles : "
    f"{len(categorical_columns_for_encoding)}"
)

print(
    "\nVariables catégorielles retenues pour l'encodage :"
)

for column in categorical_columns_for_encoding:
    print(f"  - {column}")

print(
    "\nContrôle : aucune valeur manquante n'est présente "
    "dans les variables catégorielles de X_train."
)

=== Définition des groupes de variables pour le preprocessing ===

Variables numériques : 10
Variables catégorielles : 3

Variables catégorielles retenues pour l'encodage :
  - vehicle_category_type
  - fuel_type
  - fuel_mode

Contrôle : aucune valeur manquante n'est présente dans les variables catégorielles de X_train.


## 9. Analyse du caractère nominal ou ordinal des variables catégorielles

### Objectif

Avant de définir l'encodage, cette étape examine les modalités des variables
catégorielles conservées afin de déterminer si elles possèdent ou non un ordre
métier naturel.

Une variable est considérée comme :

- **nominale** lorsque ses modalités représentent des catégories sans ordre
  intrinsèque ;
- **ordinale** lorsque ses modalités représentent des niveaux pouvant être
  classés selon un ordre métier explicite.

Les variables catégorielles conservées à ce stade sont :

- `vehicle_category_type` ;
- `fuel_type` ;
- `fuel_mode`.

L'inspection de leurs modalités permet de déterminer le type d'encodage adapté
avant la construction du pipeline de preprocessing.

In [10]:
# ---------------------------------------------------------------------
# Inspection des modalités des variables catégorielles
# ---------------------------------------------------------------------

print(
    "=== Inspection des modalités des variables catégorielles ==="
)

for column in categorical_columns:

    values = (
        X_train[column]
        .dropna()
        .astype(str)
        .unique()
    )

    values = sorted(values)

    print(
        f"\nVariable : {column}"
    )

    print(
        f"Nombre de modalités observées dans X_train : "
        f"{X_train[column].nunique(dropna=False):,}"
    )

    print(
        "Modalités : "
        + ", ".join(values)
    )

=== Inspection des modalités des variables catégorielles ===

Variable : vehicle_category_type
Nombre de modalités observées dans X_train : 2
Modalités : M1, N1

Variable : fuel_type
Nombre de modalités observées dans X_train : 9
Modalités : diesel, diesel/electric, e85, electric, hydrogen, lpg, ng, petrol, petrol/electric

Variable : fuel_mode
Nombre de modalités observées dans X_train : 6
Modalités : B, E, F, H, M, P


### 9.1 Classification nominale / ordinale

L'inspection des modalités montre qu'aucune des trois variables catégorielles
conservées ne possède un ordre métier naturel exploitable pour la modélisation :

- `vehicle_category_type` distingue les catégories réglementaires `M1` et `N1`,
  sans relation d'ordre entre elles ;
- `fuel_type` représente différents types de carburant ou d'énergie ;
- `fuel_mode` représente différents modes énergétiques.

Ces trois variables sont donc considérées comme des variables
**catégorielles nominales**.

Aucune variable catégorielle ordinale n'est identifiée parmi les variables
explicatives retenues.

In [11]:
# ---------------------------------------------------------------------
# Classification des variables catégorielles
# ---------------------------------------------------------------------

nominal_columns = [
    "vehicle_category_type",
    "fuel_type",
    "fuel_mode",
]

ordinal_columns = []


# ---------------------------------------------------------------------
# Contrôles de cohérence
# ---------------------------------------------------------------------

classified_columns = (
    nominal_columns
    + ordinal_columns
)

if set(classified_columns) != set(categorical_columns):
    raise ValueError(
        "La classification nominale / ordinale ne correspond pas "
        "aux variables catégorielles identifiées dans X_train."
    )

if set(nominal_columns).intersection(ordinal_columns):
    raise ValueError(
        "Une variable catégorielle ne peut pas être simultanément "
        "nominale et ordinale."
    )


# ---------------------------------------------------------------------
# Résumé contextualisé
# ---------------------------------------------------------------------

print(
    "=== Classification des variables catégorielles ==="
)

print(
    f"\nVariables catégorielles nominales : "
    f"{len(nominal_columns)}"
)

for column in nominal_columns:
    print(f"  - {column}")

print(
    f"\nVariables catégorielles ordinales : "
    f"{len(ordinal_columns)}"
)

if ordinal_columns:
    for column in ordinal_columns:
        print(f"  - {column}")
else:
    print("  - Aucune")

print(
    "\nConclusion : les 3 variables catégorielles retenues "
    "sont nominales et aucune variable catégorielle "
    "ordinale n'a été identifiée."
)

=== Classification des variables catégorielles ===

Variables catégorielles nominales : 3
  - vehicle_category_type
  - fuel_type
  - fuel_mode

Variables catégorielles ordinales : 0
  - Aucune

Conclusion : les 3 variables catégorielles retenues sont nominales et aucune variable catégorielle ordinale n'a été identifiée.


## 10. Analyse et traitement des valeurs manquantes

### Objectif

Avant de construire les pipelines de preprocessing, cette section analyse les
valeurs manquantes des variables explicatives.

L'objectif est de distinguer :

- les valeurs manquantes probablement accidentelles ou liées à une absence de
  saisie ;
- les valeurs manquantes pouvant avoir une signification métier ;
- les variables pour lesquelles une imputation simple est adaptée ;
- les variables pour lesquelles un indicateur binaire explicite peut être
  pertinent.

Les décisions de traitement sont prises à partir de `X_train` uniquement afin
de ne pas utiliser d'information provenant du jeu de test.

Aucune imputation n'est réalisée dans cette première étape.

### 10.1 Diagnostic quantitatif des valeurs manquantes

Cette première étape mesure la présence de valeurs manquantes dans les
variables explicatives de `X_train`.

Elle permet :

- d'identifier les variables concernées ;
- de mesurer le nombre et le pourcentage de valeurs manquantes ;
- de repérer les variables nécessitant une analyse spécifique avant
  imputation.

Ce diagnostic est uniquement quantitatif.

Il ne permet pas, à lui seul, de déterminer si une valeur est manquante pour
une raison métier ou à cause d'une absence de saisie.

Pour les variables présentant une proportion significative de valeurs
manquantes, cette distinction sera étudiée dans l'étape suivante par
croisement avec les caractéristiques métier du véhicule.

Aucune imputation ni modification de `X_train` n'est réalisée à cette étape.

In [12]:
# ---------------------------------------------------------------------
# Diagnostic quantitatif des valeurs manquantes dans X_train
# ---------------------------------------------------------------------

missing_diagnostic = []

for column in X_train.columns:

    missing_count = X_train[column].isna().sum()

    missing_pct = (
        missing_count
        / len(X_train)
        * 100
    )

    row = {
        "variable": column,
        "type": str(X_train[column].dtype),
        "valeurs_manquantes": missing_count,
        "pourcentage_manquant": round(missing_pct, 2),
        "nombre_valeurs_uniques": X_train[column].nunique(
            dropna=True
        ),
    }

    # Statistiques descriptives uniquement pour les variables numériques
    if pd.api.types.is_numeric_dtype(
        X_train[column]
    ):
        row.update(
            {
                "minimum": X_train[column].min(),
                "mediane": X_train[column].median(),
                "moyenne": X_train[column].mean(),
                "maximum": X_train[column].max(),
            }
        )

    missing_diagnostic.append(row)


# ---------------------------------------------------------------------
# Construction du tableau de diagnostic
# ---------------------------------------------------------------------

missing_diagnostic_df = (
    pd.DataFrame(missing_diagnostic)
    .sort_values(
        by="pourcentage_manquant",
        ascending=False,
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# Contrôles de cohérence
# ---------------------------------------------------------------------

if len(missing_diagnostic_df) != X_train.shape[1]:
    raise ValueError(
        "Le diagnostic ne couvre pas toutes les variables "
        "explicatives de X_train."
    )

if "manufacturer_make" in missing_diagnostic_df["variable"].values:
    raise ValueError(
        "La variable exclue 'manufacturer_make' apparaît encore "
        "dans le diagnostic."
    )


# ---------------------------------------------------------------------
# Résumé contextualisé
# ---------------------------------------------------------------------

variables_with_missing = (
    missing_diagnostic_df[
        "valeurs_manquantes"
    ] > 0
).sum()

print(
    "=== Diagnostic quantitatif des valeurs manquantes "
    "dans X_train ==="
)

print(
    f"\nNombre de variables explicatives analysées : "
    f"{X_train.shape[1]}"
)

print(
    f"Variables contenant au moins une valeur manquante : "
    f"{variables_with_missing}"
)

print(
    f"Variables sans valeur manquante : "
    f"{X_train.shape[1] - variables_with_missing}"
)

print(
    "\nLe tableau suivant présente, pour chaque variable, "
    "le nombre et le pourcentage de valeurs manquantes."
)

print(
    "Pour les variables numériques, il présente également "
    "le minimum, la médiane, la moyenne et le maximum."
)

print(
    "\nToutes les statistiques sont calculées exclusivement "
    "à partir de X_train."
)


# ---------------------------------------------------------------------
# Affichage du diagnostic
# ---------------------------------------------------------------------

display(missing_diagnostic_df)

=== Diagnostic quantitatif des valeurs manquantes dans X_train ===

Nombre de variables explicatives analysées : 13
Variables contenant au moins une valeur manquante : 7
Variables sans valeur manquante : 6

Le tableau suivant présente, pour chaque variable, le nombre et le pourcentage de valeurs manquantes.
Pour les variables numériques, il présente également le minimum, la médiane, la moyenne et le maximum.

Toutes les statistiques sont calculées exclusivement à partir de X_train.


,variable,type,valeurs_manquantes,pourcentage_manquant,nombre_valeurs_uniques,minimum,mediane,moyenne,maximum
0,electric_range_km,float64,63215,79.02,532,11.0,3.940000e+02,325.342687,796.00
1,electric_energy_consumption_wh_km,float64,63196,79.00,228,44.0,1.670000e+02,174.957570,594.00
2,co2_reduction_wltp_g_km,float64,35042,43.80,189,0.5,1.700000e+00,1.499348,2.78
3,fuel_consumption,float64,11677,14.60,146,0.3,5.500000e+00,5.498207,17.30
4,engine_capacity_cm3,float64,11355,14.19,104,875.0,1.498000e+03,1576.029922,6749.00
5,wltp_test_mass_kg,float64,225,0.28,1937,733.0,1.609000e+03,1686.848549,3093.00
6,fuel_type,object,0,0.00,9,NaN,NaN,NaN,NaN
7,mass_running_order_kg,float64,0,0.00,1339,668.0,1.502000e+03,1566.452712,3085.00
8,vehicle_category_type,object,0,0.00,2,NaN,NaN,NaN,NaN
9,fuel_mode,object,0,0.00,6,NaN,NaN,NaN,NaN


### 10.2 Analyse métier des valeurs manquantes

Le diagnostic quantitatif précédent permet d'identifier les variables
présentant des valeurs manquantes, mais il ne permet pas d'en déterminer
la cause.

Pour certaines caractéristiques techniques, une valeur manquante peut être
liée à la nature même du véhicule plutôt qu'à une erreur ou à une absence
de saisie.

Cette étape étudie donc les variables numériques contenant des valeurs
manquantes en croisant leur absence avec `fuel_type`.

L'objectif est notamment de vérifier si les valeurs manquantes sont
concentrées sur certains types de motorisation.

Cette analyse permettra ensuite de distinguer, lorsque les données le
permettent :

- les valeurs manquantes ayant vraisemblablement une origine structurelle
  ou métier ;
- les valeurs manquantes ne présentant pas de relation métier évidente
  avec le type de motorisation.

Aucune imputation et aucune modification de `X_train` ne sont réalisées
à cette étape.

In [13]:
# ---------------------------------------------------------------------
# Identification des variables numériques contenant des valeurs manquantes
# ---------------------------------------------------------------------

numeric_columns_with_missing = [
    column
    for column in numeric_columns
    if X_train[column].isna().any()
]


# ---------------------------------------------------------------------
# Contrôles préalables
# ---------------------------------------------------------------------

if "fuel_type" not in X_train.columns:
    raise ValueError(
        "La variable 'fuel_type' est nécessaire pour réaliser "
        "l'analyse métier des valeurs manquantes."
    )

if not numeric_columns_with_missing:
    raise ValueError(
        "Aucune variable numérique avec valeur manquante "
        "n'a été détectée dans X_train."
    )


# ---------------------------------------------------------------------
# Présentation générale de l'analyse
# ---------------------------------------------------------------------

print(
    "=== Analyse métier des valeurs manquantes selon le type de carburant ==="
)

print(
    f"\nVariables numériques contenant au moins une valeur manquante : "
    f"{len(numeric_columns_with_missing)}"
)

for column in numeric_columns_with_missing:
    print(f"  - {column}")

print(
    "\nPour chacune de ces variables, le tableau indique, "
    "pour chaque type de carburant :"
)

print(
    "  - le nombre total d'observations ;\n"
    "  - le nombre de valeurs manquantes ;\n"
    "  - le pourcentage de valeurs manquantes."
)

print(
    "\nCette analyse permet d'identifier si l'absence d'une "
    "caractéristique technique est concentrée sur certains "
    "types de motorisation."
)


# ---------------------------------------------------------------------
# Analyse par variable et par type de carburant
# ---------------------------------------------------------------------

missing_by_fuel_type = {}

for column in numeric_columns_with_missing:

    analysis = (
        X_train
        .groupby(
            "fuel_type",
            dropna=False,
        )
        .agg(
            observations=(column, "size"),
            valeurs_manquantes=(
                column,
                lambda series: series.isna().sum(),
            ),
        )
        .reset_index()
    )

    analysis["pourcentage_manquant"] = (
        analysis["valeurs_manquantes"]
        / analysis["observations"]
        * 100
    ).round(2)

    missing_by_fuel_type[column] = analysis


    # -----------------------------------------------------------------
    # Affichage contextualisé
    # -----------------------------------------------------------------

    print("\n" + "=" * 100)

    print(
        f"Variable analysée : {column}"
    )

    print(
        f"Valeurs manquantes dans X_train : "
        f"{X_train[column].isna().sum():,} "
        f"sur {len(X_train):,} observations "
        f"({X_train[column].isna().mean() * 100:.2f} %)"
    )

    print(
        "Répartition des valeurs manquantes "
        "selon le type de carburant :"
    )

    print("=" * 100)

    display(analysis)

=== Analyse métier des valeurs manquantes selon le type de carburant ===

Variables numériques contenant au moins une valeur manquante : 7
  - wltp_test_mass_kg
  - engine_capacity_cm3
  - engine_power_kw
  - electric_energy_consumption_wh_km
  - co2_reduction_wltp_g_km
  - fuel_consumption
  - electric_range_km

Pour chacune de ces variables, le tableau indique, pour chaque type de carburant :
  - le nombre total d'observations ;
  - le nombre de valeurs manquantes ;
  - le pourcentage de valeurs manquantes.

Cette analyse permet d'identifier si l'absence d'une caractéristique technique est concentrée sur certains types de motorisation.

Variable analysée : wltp_test_mass_kg
Valeurs manquantes dans X_train : 225 sur 80,000 observations (0.28 %)
Répartition des valeurs manquantes selon le type de carburant :


,fuel_type,observations,valeurs_manquantes,pourcentage_manquant
0,diesel,13284,26,0.20
1,diesel/electric,378,1,0.26
2,e85,445,0,0.00
3,electric,11345,102,0.90
4,hydrogen,10,0,0.00
5,lpg,1235,1,0.08
6,ng,8,0,0.00
7,petrol,48081,80,0.17
8,petrol/electric,5214,15,0.29



Variable analysée : engine_capacity_cm3
Valeurs manquantes dans X_train : 11,355 sur 80,000 observations (14.19 %)
Répartition des valeurs manquantes selon le type de carburant :


,fuel_type,observations,valeurs_manquantes,pourcentage_manquant
0,diesel,13284,0,0.0
1,diesel/electric,378,0,0.0
2,e85,445,0,0.0
3,electric,11345,11345,100.0
4,hydrogen,10,10,100.0
5,lpg,1235,0,0.0
6,ng,8,0,0.0
7,petrol,48081,0,0.0
8,petrol/electric,5214,0,0.0



Variable analysée : engine_power_kw
Valeurs manquantes dans X_train : 1 sur 80,000 observations (0.00 %)
Répartition des valeurs manquantes selon le type de carburant :


...

,fuel_type,observations,valeurs_manquantes,pourcentage_manquant
0,diesel,13284,4537,34.15
1,diesel/electric,378,378,100.00
2,e85,445,139,31.24
3,electric,11345,11345,100.00
4,hydrogen,10,10,100.00
5,lpg,1235,0,0.00
6,ng,8,6,75.00
7,petrol,48081,13413,27.90
8,petrol/electric,5214,5214,100.00



Variable analysée : fuel_consumption
Valeurs manquantes dans X_train : 11,677 sur 80,000 observations (14.60 %)
Répartition des valeurs manquantes selon le type de carburant :


,fuel_type,observations,valeurs_manquantes,pourcentage_manquant
0,diesel,13284,61,0.46
1,diesel/electric,378,1,0.26
2,e85,445,0,0.00
3,electric,11345,11345,100.00
4,hydrogen,10,10,100.00
5,lpg,1235,3,0.24
6,ng,8,8,100.00
7,petrol,48081,219,0.46
8,petrol/electric,5214,30,0.58



Variable analysée : electric_range_km
Valeurs manquantes dans X_train : 63,215 sur 80,000 observations (79.02 %)
Répartition des valeurs manquantes selon le type de carburant :


,fuel_type,observations,valeurs_manquantes,pourcentage_manquant
0,diesel,13284,13284,100.00
1,diesel/electric,378,0,0.00
2,e85,445,445,100.00
3,electric,11345,129,1.14
4,hydrogen,10,10,100.00
5,lpg,1235,1235,100.00
6,ng,8,8,100.00
7,petrol,48081,48081,100.00
8,petrol/electric,5214,23,0.44


### 10.3 Interprétation métier des valeurs manquantes

Cette section génère automatiquement l'interprétation des résultats obtenus
précédemment afin que les valeurs affichées restent cohérentes avec les données
effectivement chargées dans le notebook.

In [14]:
from IPython.display import Markdown, display


# ---------------------------------------------------------------------
# Fonctions utilitaires
# ---------------------------------------------------------------------

def get_missing_pct(column, fuel_type):
    """
    Calcule le pourcentage de valeurs manquantes d'une variable
    pour un type de motorisation donné dans X_train.
    """

    mask = X_train["fuel_type"].eq(fuel_type)

    if mask.sum() == 0:
        return None

    return (
        X_train.loc[mask, column]
        .isna()
        .mean()
        * 100
    )


def pct(column, fuel_type):
    """
    Retourne le pourcentage de valeurs manquantes formaté
    pour l'affichage Markdown.
    """

    value = get_missing_pct(
        column,
        fuel_type,
    )

    if value is None:
        return "non observé"

    return f"{value:.2f} %"


# ---------------------------------------------------------------------
# Génération dynamique de l'interprétation
# ---------------------------------------------------------------------

interpretation = f"""
#### `wltp_test_mass_kg` — Masse du véhicule lors du test WLTP

Le taux de valeurs manquantes reste très faible pour les principales
motorisations :

- diesel : **{pct("wltp_test_mass_kg", "diesel")}** ;
- diesel/électrique : **{pct("wltp_test_mass_kg", "diesel/electric")}** ;
- électrique : **{pct("wltp_test_mass_kg", "electric")}** ;
- essence : **{pct("wltp_test_mass_kg", "petrol")}** ;
- essence/électrique : **{pct("wltp_test_mass_kg", "petrol/electric")}**.

**Interprétation :** l'absence de la masse WLTP reste marginale et aucune
signification métier structurelle évidente n'est mise en évidence par le
croisement avec le type de motorisation.


#### `engine_capacity_cm3` — Cylindrée du moteur thermique

Le comportement de cette variable est particulièrement net :

- véhicules électriques :
  **{pct("engine_capacity_cm3", "electric")}** de valeurs manquantes ;
- véhicules à hydrogène :
  **{pct("engine_capacity_cm3", "hydrogen")}** de valeurs manquantes ;
- diesel : **{pct("engine_capacity_cm3", "diesel")}** ;
- essence : **{pct("engine_capacity_cm3", "petrol")}** ;
- essence/électrique :
  **{pct("engine_capacity_cm3", "petrol/electric")}**.

**Interprétation :** lorsque les valeurs manquantes sont concentrées sur les
véhicules électriques et à hydrogène, l'absence de cylindrée présente une
forte composante structurelle liée au type de motorisation.

Une imputation globale par la médiane serait alors inadaptée, car elle
attribuerait artificiellement une cylindrée de moteur thermique à des
véhicules pour lesquels cette caractéristique n'est pas renseignée.


#### `engine_power_kw` — Puissance moteur

Taux de valeurs manquantes observé pour les principales motorisations :

- diesel : **{pct("engine_power_kw", "diesel")}** ;
- électrique : **{pct("engine_power_kw", "electric")}** ;
- essence : **{pct("engine_power_kw", "petrol")}** ;
- essence/électrique :
  **{pct("engine_power_kw", "petrol/electric")}**.

**Interprétation :** lorsque ces taux restent extrêmement faibles, les valeurs
manquantes sont marginales et aucune structure métier particulière n'est mise
en évidence par cette analyse.


#### `electric_energy_consumption_wh_km` — Consommation d'énergie électrique

Taux de valeurs manquantes :

- diesel :
  **{pct("electric_energy_consumption_wh_km", "diesel")}** ;
- E85 :
  **{pct("electric_energy_consumption_wh_km", "e85")}** ;
- électrique :
  **{pct("electric_energy_consumption_wh_km", "electric")}** ;
- hydrogène :
  **{pct("electric_energy_consumption_wh_km", "hydrogen")}** ;
- GPL :
  **{pct("electric_energy_consumption_wh_km", "lpg")}** ;
- gaz naturel :
  **{pct("electric_energy_consumption_wh_km", "ng")}** ;
- essence :
  **{pct("electric_energy_consumption_wh_km", "petrol")}** ;
- diesel/électrique :
  **{pct("electric_energy_consumption_wh_km", "diesel/electric")}** ;
- essence/électrique :
  **{pct("electric_energy_consumption_wh_km", "petrol/electric")}**.

**Interprétation :** lorsque l'absence de cette information se concentre sur
les motorisations non électriques tandis que la variable est majoritairement
renseignée pour les véhicules électriques ou hybrides, les valeurs manquantes
présentent une forte composante structurelle.

Une imputation globale par la médiane serait alors peu adaptée.


#### `co2_reduction_wltp_g_km` — Réduction des émissions de CO₂ WLTP

Taux de valeurs manquantes :

- diesel :
  **{pct("co2_reduction_wltp_g_km", "diesel")}** ;
- diesel/électrique :
  **{pct("co2_reduction_wltp_g_km", "diesel/electric")}** ;
- E85 :
  **{pct("co2_reduction_wltp_g_km", "e85")}** ;
- électrique :
  **{pct("co2_reduction_wltp_g_km", "electric")}** ;
- hydrogène :
  **{pct("co2_reduction_wltp_g_km", "hydrogen")}** ;
- GPL :
  **{pct("co2_reduction_wltp_g_km", "lpg")}** ;
- gaz naturel :
  **{pct("co2_reduction_wltp_g_km", "ng")}** ;
- essence :
  **{pct("co2_reduction_wltp_g_km", "petrol")}** ;
- essence/électrique :
  **{pct("co2_reduction_wltp_g_km", "petrol/electric")}**.

**Interprétation :** la présence ou l'absence de cette variable ne doit pas
être interprétée directement comme le niveau global de réduction des émissions
de CO₂ du véhicule.

Les résultats montrent notamment que cette information peut être absente pour
l'ensemble des observations de certaines motorisations dans l'échantillon
étudié.

Cette absence systématique constitue bien une structure dans les données.
Cependant, une valeur manquante ne signifie ni que la réduction de CO₂ est
égale à zéro, ni qu'elle est nécessairement élevée.

Pour d'autres motorisations, la coexistence de valeurs présentes et manquantes
montre également que le seul `fuel_type` ne suffit pas à déterminer une valeur
de remplacement pertinente.

**Conclusion :** `co2_reduction_wltp_g_km` doit faire l'objet d'un traitement
spécifique. Une imputation globale par la médiane ne sera pas appliquée
automatiquement à cette variable.


#### `fuel_consumption` — Consommation de carburant

Taux de valeurs manquantes :

- diesel :
  **{pct("fuel_consumption", "diesel")}** ;
- diesel/électrique :
  **{pct("fuel_consumption", "diesel/electric")}** ;
- E85 :
  **{pct("fuel_consumption", "e85")}** ;
- électrique :
  **{pct("fuel_consumption", "electric")}** ;
- hydrogène :
  **{pct("fuel_consumption", "hydrogen")}** ;
- GPL :
  **{pct("fuel_consumption", "lpg")}** ;
- gaz naturel :
  **{pct("fuel_consumption", "ng")}** ;
- essence :
  **{pct("fuel_consumption", "petrol")}** ;
- essence/électrique :
  **{pct("fuel_consumption", "petrol/electric")}**.

**Interprétation :** lorsque les valeurs manquantes sont principalement
concentrées sur les motorisations pour lesquelles la consommation de carburant
n'est pas applicable ou n'est pas renseignée de la même manière, une forte
composante structurelle est mise en évidence.

Les faibles taux résiduels éventuellement observés sur les motorisations
utilisant effectivement du carburant doivent cependant être distingués de
cette absence structurelle.


#### `electric_range_km` — Autonomie électrique

Taux de valeurs manquantes :

- diesel :
  **{pct("electric_range_km", "diesel")}** ;
- diesel/électrique :
  **{pct("electric_range_km", "diesel/electric")}** ;
- E85 :
  **{pct("electric_range_km", "e85")}** ;
- électrique :
  **{pct("electric_range_km", "electric")}** ;
- hydrogène :
  **{pct("electric_range_km", "hydrogen")}** ;
- GPL :
  **{pct("electric_range_km", "lpg")}** ;
- gaz naturel :
  **{pct("electric_range_km", "ng")}** ;
- essence :
  **{pct("electric_range_km", "petrol")}** ;
- essence/électrique :
  **{pct("electric_range_km", "petrol/electric")}**.

**Interprétation :** lorsque l'autonomie électrique est absente pour les
motorisations non électriques mais majoritairement renseignée pour les
véhicules électriques et hybrides, les valeurs manquantes présentent une
forte composante structurelle.

Les éventuelles valeurs manquantes résiduelles observées chez les véhicules
électriques ou hybrides doivent néanmoins être considérées séparément.


### Synthèse

L'analyse permet de distinguer trois situations :

1. **Valeurs manquantes présentant une forte composante structurelle**
   - `engine_capacity_cm3` ;
   - `electric_energy_consumption_wh_km` ;
   - `fuel_consumption` ;
   - `electric_range_km`.

2. **Valeurs manquantes rares sans structure métier évidente mise en évidence**
   - `wltp_test_mass_kg` ;
   - `engine_power_kw`.

3. **Valeurs manquantes présentant une structure plus complexe**
   - `co2_reduction_wltp_g_km`.

Ces résultats montrent qu'une stratégie d'imputation numérique unique,
appliquée indistinctement à toutes les variables, serait insuffisante.

La prochaine étape consistera à définir, variable par variable, une stratégie
de traitement cohérente avec la nature des valeurs manquantes observées avant
la construction du pipeline de preprocessing.
"""


# ---------------------------------------------------------------------
# Affichage du Markdown généré
# ---------------------------------------------------------------------

display(
    Markdown(interpretation)
)


#### `wltp_test_mass_kg` — Masse du véhicule lors du test WLTP

Le taux de valeurs manquantes reste très faible pour les principales
motorisations :

- diesel : **0.20 %** ;
- diesel/électrique : **0.26 %** ;
- électrique : **0.90 %** ;
- essence : **0.17 %** ;
- essence/électrique : **0.29 %**.

**Interprétation :** l'absence de la masse WLTP reste marginale et aucune
signification métier structurelle évidente n'est mise en évidence par le
croisement avec le type de motorisation.


#### `engine_capacity_cm3` — Cylindrée du moteur thermique

Le comportement de cette variable est particulièrement net :

- véhicules électriques :
  **100.00 %** de valeurs manquantes ;
- véhicules à hydrogène :
  **100.00 %** de valeurs manquantes ;
- diesel : **0.00 %** ;
- essence : **0.00 %** ;
- essence/électrique :
  **0.00 %**.

**Interprétation :** lorsque les valeurs manquantes sont concentrées sur les
véhicules électriques et à hydrogène, l'absence de cylindrée présente une
forte composante structurelle liée au type de motorisation.

Une imputation globale par la médiane serait alors inadaptée, car elle
attribuerait artificiellement une cylindrée de moteur thermique à des
véhicules pour lesquels cette caractéristique n'est pas renseignée.


#### `engine_power_kw` — Puissance moteur

Taux de valeurs manquantes observé pour les principales motorisations :

- diesel : **0.00 %** ;
- électrique : **0.00 %** ;
- essence : **0.00 %** ;
- essence/électrique :
  **0.00 %**.

**Interprétation :** lorsque ces taux restent extrêmement faibles, les valeurs
manquantes sont marginales et aucune structure métier particulière n'est mise
en évidence par cette analyse.


#### `electric_energy_consumption_wh_km` — Consommation d'énergie électrique

Taux de valeurs manquantes :

- diesel :
  **100.00 %** ;
- E85 :
  **100.00 %** ;
- électrique :
  **0.96 %** ;
- hydrogène :
  **100.00 %** ;
- GPL :
  **100.00 %** ;
- gaz naturel :
  **100.00 %** ;
- essence :
  **100.00 %** ;
- diesel/électrique :
  **0.26 %** ;
- essence/électrique :
  **0.44 %**.

**Interprétation :** lorsque l'absence de cette information se concentre sur
les motorisations non électriques tandis que la variable est majoritairement
renseignée pour les véhicules électriques ou hybrides, les valeurs manquantes
présentent une forte composante structurelle.

Une imputation globale par la médiane serait alors peu adaptée.


#### `co2_reduction_wltp_g_km` — Réduction des émissions de CO₂ WLTP

Taux de valeurs manquantes :

- diesel :
  **34.15 %** ;
- diesel/électrique :
  **100.00 %** ;
- E85 :
  **31.24 %** ;
- électrique :
  **100.00 %** ;
- hydrogène :
  **100.00 %** ;
- GPL :
  **0.00 %** ;
- gaz naturel :
  **75.00 %** ;
- essence :
  **27.90 %** ;
- essence/électrique :
  **100.00 %**.

**Interprétation :** la présence ou l'absence de cette variable ne doit pas
être interprétée directement comme le niveau global de réduction des émissions
de CO₂ du véhicule.

Les résultats montrent notamment que cette information peut être absente pour
l'ensemble des observations de certaines motorisations dans l'échantillon
étudié.

Cette absence systématique constitue bien une structure dans les données.
Cependant, une valeur manquante ne signifie ni que la réduction de CO₂ est
égale à zéro, ni qu'elle est nécessairement élevée.

Pour d'autres motorisations, la coexistence de valeurs présentes et manquantes
montre également que le seul `fuel_type` ne suffit pas à déterminer une valeur
de remplacement pertinente.

**Conclusion :** `co2_reduction_wltp_g_km` doit faire l'objet d'un traitement
spécifique. Une imputation globale par la médiane ne sera pas appliquée
automatiquement à cette variable.


#### `fuel_consumption` — Consommation de carburant

Taux de valeurs manquantes :

- diesel :
  **0.46 %** ;
- diesel/électrique :
  **0.26 %** ;
- E85 :
  **0.00 %** ;
- électrique :
  **100.00 %** ;
- hydrogène :
  **100.00 %** ;
- GPL :
  **0.24 %** ;
- gaz naturel :
  **100.00 %** ;
- essence :
  **0.46 %** ;
- essence/électrique :
  **0.58 %**.

**Interprétation :** lorsque les valeurs manquantes sont principalement
concentrées sur les motorisations pour lesquelles la consommation de carburant
n'est pas applicable ou n'est pas renseignée de la même manière, une forte
composante structurelle est mise en évidence.

Les faibles taux résiduels éventuellement observés sur les motorisations
utilisant effectivement du carburant doivent cependant être distingués de
cette absence structurelle.


#### `electric_range_km` — Autonomie électrique

Taux de valeurs manquantes :

- diesel :
  **100.00 %** ;
- diesel/électrique :
  **0.00 %** ;
- E85 :
  **100.00 %** ;
- électrique :
  **1.14 %** ;
- hydrogène :
  **100.00 %** ;
- GPL :
  **100.00 %** ;
- gaz naturel :
  **100.00 %** ;
- essence :
  **100.00 %** ;
- essence/électrique :
  **0.44 %**.

**Interprétation :** lorsque l'autonomie électrique est absente pour les
motorisations non électriques mais majoritairement renseignée pour les
véhicules électriques et hybrides, les valeurs manquantes présentent une
forte composante structurelle.

Les éventuelles valeurs manquantes résiduelles observées chez les véhicules
électriques ou hybrides doivent néanmoins être considérées séparément.


### Synthèse

L'analyse permet de distinguer trois situations :

1. **Valeurs manquantes présentant une forte composante structurelle**
   - `engine_capacity_cm3` ;
   - `electric_energy_consumption_wh_km` ;
   - `fuel_consumption` ;
   - `electric_range_km`.

2. **Valeurs manquantes rares sans structure métier évidente mise en évidence**
   - `wltp_test_mass_kg` ;
   - `engine_power_kw`.

3. **Valeurs manquantes présentant une structure plus complexe**
   - `co2_reduction_wltp_g_km`.

Ces résultats montrent qu'une stratégie d'imputation numérique unique,
appliquée indistinctement à toutes les variables, serait insuffisante.

La prochaine étape consistera à définir, variable par variable, une stratégie
de traitement cohérente avec la nature des valeurs manquantes observées avant
la construction du pipeline de preprocessing.


### 10.4 Définition des stratégies de traitement des valeurs manquantes

Les analyses précédentes montrent que toutes les valeurs manquantes ne
possèdent pas la même signification.

Une imputation uniforme de toutes les variables numériques par leur médiane
n'est donc pas retenue.

Pour certaines variables techniques, deux types de valeurs manquantes doivent
être distingués :

- **NaN structurel** : la caractéristique n'est pas applicable au véhicule
  considéré. Dans ce cas, l'absence de valeur possède une signification métier.
- **NaN résiduel** : la caractéristique est applicable au véhicule, mais sa
  valeur reste manquante. Il s'agit alors d'une donnée absente qui nécessite
  une imputation.

Lorsqu'un NaN est structurel et que la valeur `0` possède une interprétation
métier cohérente, le NaN peut être remplacé par `0`.

Lorsqu'un NaN est résiduel, une imputation statistique est réalisée. Lorsque
cela est pertinent, un indicateur binaire `has_...` permet également de
conserver l'information selon laquelle la valeur originale était présente
ou absente.

Les statistiques nécessaires aux imputations seront apprises exclusivement
sur `X_train`.


#### 1. `engine_capacity_cm3` — Cylindrée du moteur thermique

La cylindrée correspond à la capacité du moteur thermique.

L'analyse précédente montre que les valeurs manquantes sont concentrées sur
les véhicules électriques et à hydrogène, tandis qu'aucune valeur manquante
n'est observée pour les autres motorisations dans l'échantillon étudié.

Pour les véhicules électriques et à hydrogène, l'absence de cylindrée est
considérée comme **structurelle**.

**Décision :**

- pour une motorisation pour laquelle la cylindrée thermique n'est pas
  applicable, remplacer le NaN par `0` ;
- si de futurs jeux de données présentent un NaN pour une motorisation
  thermique, celui-ci sera considéré comme **résiduel** et fera l'objet
  d'une imputation statistique ;
- aucun indicateur binaire supplémentaire n'est créé à ce stade, car
  `fuel_type` permet déjà d'identifier la nature de la motorisation.

La valeur `0` représente ici l'absence de cylindrée thermique et non une
cylindrée inconnue.


#### 2. `electric_energy_consumption_wh_km` — Consommation d'énergie électrique

Cette variable mesure la consommation d'énergie électrique du véhicule.

L'analyse montre que son absence est systématique pour les motorisations
non électriques, tandis que la variable est presque toujours renseignée
pour les véhicules électriques ou hybrides.

Deux situations doivent donc être distinguées :

- pour une motorisation non électrique, le NaN est **structurel** ;
- pour un véhicule électrique ou hybride, le NaN est **résiduel**.

**Décision :**

- remplacer les NaN structurels par `0` ;
- imputer statistiquement les NaN résiduels des véhicules électriques
  ou hybrides ;
- créer un indicateur binaire `has_electric_energy_consumption` permettant,
  pour les véhicules auxquels cette caractéristique est applicable, de
  conserver l'information selon laquelle la valeur originale était présente
  ou manquante.

La valeur `0` représente l'absence de consommation électrique applicable
à la motorisation.

Elle ne doit pas être utilisée pour représenter une consommation électrique
inconnue d'un véhicule électrique ou hybride.


#### 3. `electric_range_km` — Autonomie électrique

Cette variable représente l'autonomie électrique du véhicule.

Une motorisation ne disposant pas d'une capacité de déplacement électrique
n'a pas d'autonomie électrique applicable.

L'analyse confirme cette structure : les motorisations non électriques
présentent une absence systématique de cette information, alors que la
variable est presque toujours renseignée pour les véhicules électriques
et hybrides.

Deux situations sont donc distinguées :

- pour une motorisation non électrique, le NaN est **structurel** ;
- pour un véhicule électrique ou hybride, le NaN est **résiduel**.

**Décision :**

- remplacer les NaN structurels par `0` ;
- imputer statistiquement les NaN résiduels des véhicules électriques
  ou hybrides ;
- créer un indicateur binaire `has_electric_range` afin de conserver
  l'information selon laquelle l'autonomie électrique originale était
  présente ou manquante lorsque cette caractéristique est applicable.

Il ne serait pas cohérent d'attribuer à un véhicule purement thermique
l'autonomie électrique médiane d'un véhicule électrique.

Inversement, remplacer par `0` une autonomie manquante d'un véhicule
électrique reviendrait à lui attribuer artificiellement une autonomie
électrique de 0 km. Ces NaN résiduels doivent donc être imputés.


#### 4. `fuel_consumption` — Consommation de carburant

Cette variable présente également deux situations différentes.

Pour certaines motorisations, l'absence de consommation de carburant est
liée à la nature du véhicule. Pour d'autres motorisations utilisant
effectivement du carburant, de faibles proportions de valeurs manquantes
subsistent.

Deux types de NaN sont donc distingués :

- **NaN structurel** lorsque la consommation de carburant n'est pas
  applicable à la motorisation ;
- **NaN résiduel** lorsque le véhicule utilise un carburant mais que sa
  consommation n'est pas renseignée.

**Décision :**

- remplacer les NaN structurels par `0` ;
- imputer statistiquement les NaN résiduels uniquement parmi les
  motorisations pour lesquelles la consommation de carburant est applicable ;
- créer un indicateur binaire `has_fuel_consumption` permettant de conserver
  l'information selon laquelle la consommation originale était présente ou
  manquante lorsque cette caractéristique est applicable.

Cette distinction évite d'attribuer artificiellement une consommation de
carburant à un véhicule pour lequel cette caractéristique n'est pas
applicable.


#### 5. `mass_running_order_kg` — Masse en ordre de marche

La validation du preprocessing sur le dataset complet a mis en évidence
quelques valeurs manquantes de `mass_running_order_kg`, qui n'étaient pas
présentes dans l'échantillon utilisé lors de l'analyse initiale.

La masse en ordre de marche constitue une caractéristique applicable à
l'ensemble des véhicules. Son absence ne correspond donc pas à une
non-applicabilité métier.

Ces valeurs manquantes sont considérées comme **résiduelles**.

**Décision :**

- appliquer une imputation par la médiane calculée exclusivement à partir
  de `X_train` ;
- appliquer cette même médiane à `X_train` et `X_test` ;
- ne pas créer d'indicateur binaire spécifique compte tenu du caractère
  rare de ces valeurs manquantes.

La médiane est retenue afin de limiter la sensibilité de l'imputation aux
valeurs extrêmes.


#### 6. `wltp_test_mass_kg` — Masse du véhicule lors du test WLTP

Les valeurs manquantes sont rares et aucune justification métier structurelle
n'a été mise en évidence par l'analyse précédente.

La masse WLTP est une caractéristique applicable au véhicule indépendamment
du type de motorisation.

Les valeurs manquantes sont donc considérées comme **résiduelles**.

**Décision :**

- appliquer une imputation statistique par la médiane calculée exclusivement
  à partir de `X_train` ;
- ne pas créer d'indicateur binaire spécifique compte tenu du faible taux
  de valeurs manquantes et de l'absence de structure métier mise en évidence.

La médiane est retenue afin de limiter la sensibilité de l'imputation aux
valeurs extrêmes.


#### 7. `engine_power_kw` — Puissance moteur

Les valeurs manquantes observées sont extrêmement rares et aucune structure
métier particulière n'a été identifiée.

La puissance moteur reste une caractéristique applicable aux véhicules
concernés.

Les valeurs manquantes sont donc considérées comme **résiduelles**.

**Décision :**

- appliquer une imputation par la médiane calculée exclusivement à partir
  de `X_train` ;
- ne pas créer d'indicateur binaire spécifique compte tenu du caractère
  exceptionnel des valeurs manquantes.


#### 8. `co2_reduction_wltp_g_km` — Réduction des émissions de CO₂ WLTP

Cette variable présente une proportion importante de valeurs manquantes et
une structure particulière.

Une imputation globale par la moyenne ou la médiane n'est pas retenue, car
elle attribuerait artificiellement une valeur de réduction de CO₂ aux
observations pour lesquelles aucune réduction n'est renseignée dans cette
variable.

L'absence de valeur constitue elle-même une information potentiellement
utile au modèle.

**Décision :**

- créer une variable binaire `has_co2_reduction_wltp` :
  - `1` lorsque la valeur originale de `co2_reduction_wltp_g_km`
    est renseignée ;
  - `0` lorsque la valeur originale est manquante ;
- remplacer ensuite les valeurs manquantes de
  `co2_reduction_wltp_g_km` par `0`.

La valeur `0` utilisée pour l'imputation ne signifie pas nécessairement
que le véhicule ne bénéficie physiquement d'aucune réduction de ses
émissions de CO₂.

Elle représente l'absence de réduction renseignée dans cette variable.

L'indicateur `has_co2_reduction_wltp` permet de distinguer une valeur
réellement renseignée d'une valeur créée lors du traitement des données.


### Synthèse des décisions

| Variable | Nature des NaN | Traitement retenu | Indicateur |
|---|---|---|---|
| `engine_capacity_cm3` | Structurelle si cylindrée non applicable ; éventuelle absence résiduelle sinon | Structurel → `0` ; résiduel → imputation statistique | Aucun à ce stade |
| `electric_energy_consumption_wh_km` | Structurelle pour les motorisations non électriques ; résiduelle pour les électriques/hybrides | Structurel → `0` ; résiduel → imputation statistique | `has_electric_energy_consumption` |
| `electric_range_km` | Structurelle pour les motorisations non électriques ; résiduelle pour les électriques/hybrides | Structurel → `0` ; résiduel → imputation statistique | `has_electric_range` |
| `fuel_consumption` | Structurelle lorsque non applicable ; résiduelle lorsqu'elle est applicable | Structurel → `0` ; résiduel → imputation statistique | `has_fuel_consumption` |
| `mass_running_order_kg` | Rare et résiduelle | Médiane apprise sur `X_train` | Aucun |
| `wltp_test_mass_kg` | Rare et résiduelle | Médiane apprise sur `X_train` | Aucun |
| `engine_power_kw` | Exceptionnelle et résiduelle | Médiane apprise sur `X_train` | Aucun |
| `co2_reduction_wltp_g_km` | Absence informative / structure particulière | NaN → `0` | `has_co2_reduction_wltp` |


### Principe général retenu

La stratégie de preprocessing distingue désormais trois situations.

**1. NaN structurel**

La caractéristique n'est pas applicable au véhicule.

Lorsque la valeur `0` possède une interprétation métier cohérente, elle est
utilisée pour représenter cette non-applicabilité.

**2. NaN résiduel**

La caractéristique est applicable au véhicule, mais sa valeur n'a pas été
renseignée.

Une imputation statistique est alors réalisée à partir de `X_train`.

Pour les variables où cette absence peut constituer une information utile,
un indicateur `has_...` permet de conserver la connaissance de l'état
original de la donnée.

**3. Absence informative particulière**

Pour `co2_reduction_wltp_g_km`, la valeur manquante est remplacée par `0`
tout en conservant explicitement son absence originale au moyen de
`has_co2_reduction_wltp`.

### 10.5 Création des indicateurs binaires de présence

Conformément aux décisions prises au point 10.4, un indicateur binaire est
créé pour les variables dont l'absence d'information doit être conservée
comme information supplémentaire pour le modèle.

Pour chaque variable concernée :

- `1` indique que la valeur était présente dans la donnée originale ;
- `0` indique que la valeur était manquante avant traitement.

Les indicateurs sont créés **avant toute imputation**, afin de préserver
l'information originale sur la présence ou l'absence de la donnée.

Les indicateurs retenus sont :

- `has_electric_energy_consumption_wh_km` ;
- `has_electric_range_km` ;
- `has_fuel_consumption` ;
- `has_co2_reduction_wltp_g_km`.

Aucun indicateur supplémentaire n'est créé pour les autres variables,
conformément aux décisions du point 10.4.

In [15]:
# ---------------------------------------------------------------------
# 10.5 - Création des indicateurs binaires de présence
# ---------------------------------------------------------------------

# Copies de travail utilisées pour les traitements à venir.
# Les jeux X_train et X_test d'origine restent ainsi inchangés.
X_train_processed = X_train.copy()
X_test_processed = X_test.copy()


# ---------------------------------------------------------------------
# Définition des indicateurs
# ---------------------------------------------------------------------

# Correspondance entre chaque variable source et l'indicateur
# permettant de mémoriser si sa valeur était initialement présente.
indicator_columns = {
    "electric_energy_consumption_wh_km":
        "has_electric_energy_consumption_wh_km",

    "electric_range_km":
        "has_electric_range_km",

    "fuel_consumption":
        "has_fuel_consumption",

    "co2_reduction_wltp_g_km":
        "has_co2_reduction_wltp_g_km",
}


# ---------------------------------------------------------------------
# Création des indicateurs AVANT toute imputation
# ---------------------------------------------------------------------
#
# 1 -> valeur présente dans la donnée originale
# 0 -> valeur manquante dans la donnée originale
# ---------------------------------------------------------------------

for source_column, indicator_column in indicator_columns.items():

    X_train_processed[indicator_column] = (
        X_train_processed[source_column]
        .notna()
        .astype("int8")
    )

    X_test_processed[indicator_column] = (
        X_test_processed[source_column]
        .notna()
        .astype("int8")
    )


# ---------------------------------------------------------------------
# Contrôles de cohérence
# ---------------------------------------------------------------------

for source_column, indicator_column in indicator_columns.items():

    # Les indicateurs doivent uniquement contenir 0 et/ou 1.
    train_values = set(
        X_train_processed[indicator_column].unique()
    )

    test_values = set(
        X_test_processed[indicator_column].unique()
    )

    if not train_values.issubset({0, 1}):
        raise ValueError(
            f"L'indicateur '{indicator_column}' contient "
            "des valeurs inattendues dans X_train."
        )

    if not test_values.issubset({0, 1}):
        raise ValueError(
            f"L'indicateur '{indicator_column}' contient "
            "des valeurs inattendues dans X_test."
        )

    # Le nombre de 0 doit correspondre exactement au nombre
    # de valeurs manquantes de la variable source avant imputation.
    train_missing = X_train[source_column].isna().sum()
    test_missing = X_test[source_column].isna().sum()

    train_indicator_zero = (
        X_train_processed[indicator_column] == 0
    ).sum()

    test_indicator_zero = (
        X_test_processed[indicator_column] == 0
    ).sum()

    if train_missing != train_indicator_zero:
        raise ValueError(
            f"Incohérence détectée pour '{indicator_column}' "
            "dans X_train."
        )

    if test_missing != test_indicator_zero:
        raise ValueError(
            f"Incohérence détectée pour '{indicator_column}' "
            "dans X_test."
        )


# ---------------------------------------------------------------------
# Résumé
# ---------------------------------------------------------------------

print(
    "=== Création des indicateurs binaires de présence ==="
)

print(
    "\nLes indicateurs sont créés avant toute imputation afin "
    "de conserver l'information indiquant si la valeur "
    "originale était présente ou manquante."
)

print(
    f"\nNombre d'indicateurs créés : "
    f"{len(indicator_columns)}"
)

for source_column, indicator_column in indicator_columns.items():

    train_present = int(
        X_train_processed[indicator_column].sum()
    )

    train_missing = (
        len(X_train_processed) - train_present
    )

    test_present = int(
        X_test_processed[indicator_column].sum()
    )

    test_missing = (
        len(X_test_processed) - test_present
    )

    print(
        f"\nIndicateur : {indicator_column}"
    )

    print(
        f"  Variable source : {source_column}"
    )

    print(
        f"  X_train : "
        f"{train_present:,} valeurs initialement présentes | "
        f"{train_missing:,} valeurs initialement manquantes"
    )

    print(
        f"  X_test  : "
        f"{test_present:,} valeurs initialement présentes | "
        f"{test_missing:,} valeurs initialement manquantes"
    )


print(
    "\nContrôle de cohérence : "
    "tous les indicateurs correspondent aux valeurs "
    "présentes/manquantes des données originales."
)

print(
    f"\nNombre de variables explicatives avant création "
    f"des indicateurs : {X_train.shape[1]}"
)

print(
    f"Nombre de variables explicatives après création "
    f"des indicateurs : {X_train_processed.shape[1]}"
)

=== Création des indicateurs binaires de présence ===

Les indicateurs sont créés avant toute imputation afin de conserver l'information indiquant si la valeur originale était présente ou manquante.

Nombre d'indicateurs créés : 4

Indicateur : has_electric_energy_consumption_wh_km
  Variable source : electric_energy_consumption_wh_km
  X_train : 16,804 valeurs initialement présentes | 63,196 valeurs initialement manquantes
  X_test  : 4,234 valeurs initialement présentes | 15,766 valeurs initialement manquantes

Indicateur : has_electric_range_km
  Variable source : electric_range_km
  X_train : 16,785 valeurs initialement présentes | 63,215 valeurs initialement manquantes
  X_test  : 4,228 valeurs initialement présentes | 15,772 valeurs initialement manquantes

Indicateur : has_fuel_consumption
  Variable source : fuel_consumption
  X_train : 68,323 valeurs initialement présentes | 11,677 valeurs initialement manquantes
  X_test  : 17,068 valeurs initialement présentes | 2,932 valeur

### 10.6 Traitement conditionnel des NaN structurels et résiduels

Conformément aux décisions prises au point 10.4, les valeurs manquantes
des variables suivantes nécessitent un traitement conditionnel :

- `engine_capacity_cm3` ;
- `electric_energy_consumption_wh_km` ;
- `electric_range_km` ;
- `fuel_consumption`.

Deux situations sont distinguées :

1. **NaN structurel**

   La variable n'est pas applicable au type de motorisation concerné.

   Dans ce cas :

   - la valeur manquante est remplacée par `0` ;
   - aucune médiane n'est utilisée pour cette observation.

2. **NaN résiduel**

   La variable est applicable au véhicule, mais sa valeur est absente.

   Dans ce cas :

   - la médiane est calculée exclusivement à partir des valeurs pertinentes
     et présentes de `X_train` ;
   - cette médiane est utilisée pour imputer les NaN résiduels de `X_train` ;
   - la même médiane apprise sur `X_train` est utilisée pour les NaN
     résiduels de `X_test`.

Les indicateurs binaires créés au point 10.5 restent inchangés et conservent
l'information indiquant si la valeur était initialement présente ou absente.

Cette étape ne traite pas encore :

- `co2_reduction_wltp_g_km` ;
- `wltp_test_mass_kg` ;
- `engine_power_kw`.

Ces variables feront l'objet des étapes suivantes.

In [16]:
# ---------------------------------------------------------------------
# 10.6 - Traitement conditionnel des NaN structurels et résiduels
# ---------------------------------------------------------------------

# Variables électriques applicables aux véhicules électriques
# et hybrides.
electric_fuel_types = {
    "electric",
    "diesel/electric",
    "petrol/electric",
}

# Cylindrée applicable aux véhicules possédant un moteur thermique.
thermal_engine_fuel_types = {
    "diesel",
    "diesel/electric",
    "e85",
    "lpg",
    "ng",
    "petrol",
    "petrol/electric",
}

# Consommation de carburant applicable aux motorisations utilisant
# effectivement un carburant.
fuel_consuming_types = {
    "diesel",
    "diesel/electric",
    "e85",
    "lpg",
    "petrol",
    "petrol/electric",
}


# ---------------------------------------------------------------------
# Fonction de traitement conditionnel
# ---------------------------------------------------------------------

def apply_conditional_imputation(
    train_df,
    test_df,
    column,
    applicable_fuel_types,
):
    """
    Traite les NaN d'une variable selon les décisions du point 10.4.

    NaN structurel :
        variable non applicable à la motorisation -> remplacement par 0.

    NaN résiduel :
        variable applicable mais valeur absente -> remplacement par
        la médiane calculée exclusivement sur X_train.

    La médiane apprise sur X_train est également utilisée pour X_test.
    """

    # La variable est-elle applicable à la motorisation ?
    train_applicable = train_df["fuel_type"].isin(
        applicable_fuel_types
    )

    test_applicable = test_df["fuel_type"].isin(
        applicable_fuel_types
    )


    # -------------------------------------------------------------
    # Identification des NaN AVANT traitement
    # -------------------------------------------------------------

    train_structural_mask = (
        ~train_applicable
        & train_df[column].isna()
    )

    train_residual_mask = (
        train_applicable
        & train_df[column].isna()
    )

    test_structural_mask = (
        ~test_applicable
        & test_df[column].isna()
    )

    test_residual_mask = (
        test_applicable
        & test_df[column].isna()
    )


    # -------------------------------------------------------------
    # Calcul de la médiane exclusivement sur X_train
    # -------------------------------------------------------------

    train_reference_values = train_df.loc[
        train_applicable
        & train_df[column].notna(),
        column,
    ]

    if train_reference_values.empty:
        raise ValueError(
            f"Aucune valeur pertinente disponible dans X_train "
            f"pour calculer la médiane de '{column}'."
        )

    median_train = train_reference_values.median()


    # -------------------------------------------------------------
    # Traitement des NaN structurels
    # -------------------------------------------------------------

    train_df.loc[
        train_structural_mask,
        column,
    ] = 0.0

    test_df.loc[
        test_structural_mask,
        column,
    ] = 0.0


    # -------------------------------------------------------------
    # Traitement des NaN résiduels
    # -------------------------------------------------------------

    train_df.loc[
        train_residual_mask,
        column,
    ] = median_train

    test_df.loc[
        test_residual_mask,
        column,
    ] = median_train


    # -------------------------------------------------------------
    # Contrôle après traitement
    # -------------------------------------------------------------

    train_remaining_nan = int(
        train_df[column].isna().sum()
    )

    test_remaining_nan = int(
        test_df[column].isna().sum()
    )

    if train_remaining_nan != 0:
        raise ValueError(
            f"Des valeurs manquantes subsistent dans "
            f"'{column}' pour X_train."
        )

    if test_remaining_nan != 0:
        raise ValueError(
            f"Des valeurs manquantes subsistent dans "
            f"'{column}' pour X_test."
        )


    # -------------------------------------------------------------
    # Résultat du traitement
    # -------------------------------------------------------------

    return {
        "variable": column,
        "median_train": median_train,

        "train_structural_nan": int(
            train_structural_mask.sum()
        ),

        "train_residual_nan": int(
            train_residual_mask.sum()
        ),

        "test_structural_nan": int(
            test_structural_mask.sum()
        ),

        "test_residual_nan": int(
            test_residual_mask.sum()
        ),

        "train_remaining_nan": train_remaining_nan,
        "test_remaining_nan": test_remaining_nan,
    }


# ---------------------------------------------------------------------
# Application aux quatre variables concernées
# ---------------------------------------------------------------------

conditional_imputation_report = []


conditional_imputation_report.append(
    apply_conditional_imputation(
        train_df=X_train_processed,
        test_df=X_test_processed,
        column="engine_capacity_cm3",
        applicable_fuel_types=thermal_engine_fuel_types,
    )
)


conditional_imputation_report.append(
    apply_conditional_imputation(
        train_df=X_train_processed,
        test_df=X_test_processed,
        column="electric_energy_consumption_wh_km",
        applicable_fuel_types=electric_fuel_types,
    )
)


conditional_imputation_report.append(
    apply_conditional_imputation(
        train_df=X_train_processed,
        test_df=X_test_processed,
        column="electric_range_km",
        applicable_fuel_types=electric_fuel_types,
    )
)


conditional_imputation_report.append(
    apply_conditional_imputation(
        train_df=X_train_processed,
        test_df=X_test_processed,
        column="fuel_consumption",
        applicable_fuel_types=fuel_consuming_types,
    )
)


# ---------------------------------------------------------------------
# Construction du tableau de contrôle
# ---------------------------------------------------------------------

conditional_imputation_report_df = pd.DataFrame(
    conditional_imputation_report
)


# ---------------------------------------------------------------------
# Résumé contextualisé
# ---------------------------------------------------------------------

print(
    "=== Traitement conditionnel des NaN "
    "structurels et résiduels ==="
)

print(
    "\nNombre de variables traitées : "
    f"{len(conditional_imputation_report_df)}"
)

print(
    "\nPour chaque variable :"
)

print(
    "  - les NaN structurels ont été remplacés par 0 ;"
)

print(
    "  - les NaN résiduels ont été remplacés par "
    "la médiane calculée uniquement sur X_train ;"
)

print(
    "  - la médiane apprise sur X_train a été réutilisée "
    "sans recalcul sur X_test."
)

print(
    "\nLe tableau suivant présente les valeurs apprises "
    "et le nombre d'observations traitées."
)


# ---------------------------------------------------------------------
# Affichage du rapport
# ---------------------------------------------------------------------

display(conditional_imputation_report_df)


# ---------------------------------------------------------------------
# Contrôle global
# ---------------------------------------------------------------------

treated_columns = [
    "engine_capacity_cm3",
    "electric_energy_consumption_wh_km",
    "electric_range_km",
    "fuel_consumption",
]

train_remaining_nan = (
    X_train_processed[treated_columns]
    .isna()
    .sum()
    .sum()
)

test_remaining_nan = (
    X_test_processed[treated_columns]
    .isna()
    .sum()
    .sum()
)

if train_remaining_nan != 0:
    raise ValueError(
        "Des valeurs manquantes subsistent dans les variables "
        "traitées de X_train."
    )

if test_remaining_nan != 0:
    raise ValueError(
        "Des valeurs manquantes subsistent dans les variables "
        "traitées de X_test."
    )

print(
    "\nContrôle final : aucun NaN ne subsiste dans "
    "les quatre variables traitées, ni dans X_train_processed "
    "ni dans X_test_processed."
)

=== Traitement conditionnel des NaN structurels et résiduels ===

Nombre de variables traitées : 4

Pour chaque variable :
  - les NaN structurels ont été remplacés par 0 ;
  - les NaN résiduels ont été remplacés par la médiane calculée uniquement sur X_train ;
  - la médiane apprise sur X_train a été réutilisée sans recalcul sur X_test.

Le tableau suivant présente les valeurs apprises et le nombre d'observations traitées.


,variable,median_train,train_structural_nan,train_residual_nan,test_structural_nan,test_residual_nan,train_remaining_nan,test_remaining_nan
0,engine_capacity_cm3,1498.0,11355,0,2840,0,0,0
1,electric_energy_consumption_wh_km,167.0,63063,133,15733,33,0,0
2,electric_range_km,394.0,63063,152,15733,39,0,0
3,fuel_consumption,5.5,11363,314,2842,90,0,0



Contrôle final : aucun NaN ne subsiste dans les quatre variables traitées, ni dans X_train_processed ni dans X_test_processed.


### 10.7 Traitement de `co2_reduction_wltp_g_km`

Conformément à la décision prise au point 10.4, la variable
`co2_reduction_wltp_g_km` fait l'objet d'un traitement spécifique.

Pour cette variable, l'absence de valeur est considérée comme une information
particulière qui ne doit pas être traitée par une imputation statistique.

La stratégie retenue est donc :

- les valeurs présentes sont conservées telles quelles ;
- les valeurs `NaN` sont remplacées par `0` ;
- aucune moyenne ou médiane n'est calculée ;
- l'information indiquant si la valeur était initialement présente ou absente
  est déjà conservée dans `has_co2_reduction_wltp_g_km`, créé au point 10.5.

Ainsi, la valeur numérique traitée et l'information sur sa présence initiale
restent disponibles séparément pour le futur modèle.

In [17]:
# ---------------------------------------------------------------------
# 10.7 - Traitement spécifique de co2_reduction_wltp_g_km
# ---------------------------------------------------------------------

column = "co2_reduction_wltp_g_km"
indicator_column = "has_co2_reduction_wltp_g_km"


# ---------------------------------------------------------------------
# 1. Contrôles préalables
# ---------------------------------------------------------------------

if column not in X_train_processed.columns:
    raise ValueError(
        f"La variable '{column}' est absente de X_train_processed."
    )

if column not in X_test_processed.columns:
    raise ValueError(
        f"La variable '{column}' est absente de X_test_processed."
    )

if indicator_column not in X_train_processed.columns:
    raise ValueError(
        f"L'indicateur '{indicator_column}' est absent "
        "de X_train_processed."
    )

if indicator_column not in X_test_processed.columns:
    raise ValueError(
        f"L'indicateur '{indicator_column}' est absent "
        "de X_test_processed."
    )


# ---------------------------------------------------------------------
# 2. Identification des NaN AVANT traitement
# ---------------------------------------------------------------------

train_missing_mask = (
    X_train_processed[column]
    .isna()
)

test_missing_mask = (
    X_test_processed[column]
    .isna()
)

train_missing_count = int(
    train_missing_mask.sum()
)

test_missing_count = int(
    test_missing_mask.sum()
)


# ---------------------------------------------------------------------
# 3. Vérification de la cohérence avec l'indicateur
# ---------------------------------------------------------------------

train_indicator_missing_count = int(
    (
        X_train_processed[indicator_column] == 0
    ).sum()
)

test_indicator_missing_count = int(
    (
        X_test_processed[indicator_column] == 0
    ).sum()
)

if train_missing_count != train_indicator_missing_count:
    raise ValueError(
        "Incohérence entre les NaN de "
        f"'{column}' et l'indicateur '{indicator_column}' "
        "dans X_train_processed."
    )

if test_missing_count != test_indicator_missing_count:
    raise ValueError(
        "Incohérence entre les NaN de "
        f"'{column}' et l'indicateur '{indicator_column}' "
        "dans X_test_processed."
    )


# ---------------------------------------------------------------------
# 4. Application de la décision du point 10.4
# ---------------------------------------------------------------------
#
# NaN -> 0
#
# Aucune moyenne ou médiane n'est calculée.
# L'indicateur créé au point 10.5 conserve l'information
# indiquant si la valeur était initialement présente.
# ---------------------------------------------------------------------

X_train_processed.loc[
    train_missing_mask,
    column,
] = 0.0

X_test_processed.loc[
    test_missing_mask,
    column,
] = 0.0


# ---------------------------------------------------------------------
# 5. Vérification après traitement
# ---------------------------------------------------------------------

train_remaining_nan = int(
    X_train_processed[column]
    .isna()
    .sum()
)

test_remaining_nan = int(
    X_test_processed[column]
    .isna()
    .sum()
)

if train_remaining_nan != 0:
    raise ValueError(
        f"Des NaN subsistent dans '{column}' "
        "pour X_train_processed."
    )

if test_remaining_nan != 0:
    raise ValueError(
        f"Des NaN subsistent dans '{column}' "
        "pour X_test_processed."
    )


# ---------------------------------------------------------------------
# 6. Rapport du traitement effectué
# ---------------------------------------------------------------------

co2_reduction_report_df = pd.DataFrame(
    [
        {
            "dataset": "Train",
            "nan_avant_traitement": train_missing_count,
            "traitement": "NaN -> 0",
            "valeur_utilisee": 0.0,
            "nan_apres_traitement": train_remaining_nan,
        },
        {
            "dataset": "Test",
            "nan_avant_traitement": test_missing_count,
            "traitement": "NaN -> 0",
            "valeur_utilisee": 0.0,
            "nan_apres_traitement": test_remaining_nan,
        },
    ]
)


# ---------------------------------------------------------------------
# 7. Résumé contextualisé
# ---------------------------------------------------------------------

print(
    "=== Traitement spécifique de "
    "co2_reduction_wltp_g_km ==="
)

print(
    "\nCette variable n'est pas imputée par une statistique."
)

print(
    "Les valeurs manquantes sont remplacées par 0, "
    "tandis que l'indicateur "
    "'has_co2_reduction_wltp_g_km' conserve "
    "l'information sur leur absence initiale."
)

print(
    f"\nX_train_processed : "
    f"{train_missing_count:,} NaN remplacés par 0"
)

print(
    f"X_test_processed  : "
    f"{test_missing_count:,} NaN remplacés par 0"
)

print(
    "\nContrôle : aucun NaN ne subsiste dans "
    "'co2_reduction_wltp_g_km'."
)


# ---------------------------------------------------------------------
# Affichage du rapport
# ---------------------------------------------------------------------

display(co2_reduction_report_df)

=== Traitement spécifique de co2_reduction_wltp_g_km ===

Cette variable n'est pas imputée par une statistique.
Les valeurs manquantes sont remplacées par 0, tandis que l'indicateur 'has_co2_reduction_wltp_g_km' conserve l'information sur leur absence initiale.

X_train_processed : 35,042 NaN remplacés par 0
X_test_processed  : 8,725 NaN remplacés par 0

Contrôle : aucun NaN ne subsiste dans 'co2_reduction_wltp_g_km'.


,dataset,nan_avant_traitement,traitement,valeur_utilisee,nan_apres_traitement
0,Train,35042,NaN -> 0,0.0,0
1,Test,8725,NaN -> 0,0.0,0


### 10.8 Imputation des valeurs manquantes rares

Conformément aux décisions prises au point 10.4, les variables
`wltp_test_mass_kg` et `engine_power_kw` présentent un faible nombre de
valeurs manquantes résiduelles.

Aucune structure métier particulière n'a été mise en évidence pour ces NaN.

La stratégie retenue est donc une imputation par la médiane :

- la médiane est calculée exclusivement sur `X_train_processed` ;
- les NaN de `X_train_processed` sont remplacés par cette médiane ;
- la même médiane est ensuite appliquée aux NaN de `X_test_processed` ;
- aucune statistique n'est recalculée sur le jeu de test.

Cette étape concerne uniquement `wltp_test_mass_kg` et `engine_power_kw`.

`mass_running_order_kg` ne nécessite aucune imputation, car aucune valeur
manquante n'a été détectée pour cette variable.

In [18]:
# ---------------------------------------------------------------------
# 10.8 - Imputation des valeurs manquantes rares
# ---------------------------------------------------------------------

rare_missing_columns = [
    "wltp_test_mass_kg",
    "engine_power_kw",
]

rare_imputation_report = []


# ---------------------------------------------------------------------
# Contrôles préalables
# ---------------------------------------------------------------------

for column in rare_missing_columns:

    if column not in X_train_processed.columns:
        raise ValueError(
            f"La variable '{column}' est absente de X_train_processed."
        )

    if column not in X_test_processed.columns:
        raise ValueError(
            f"La variable '{column}' est absente de X_test_processed."
        )


# ---------------------------------------------------------------------
# Imputation variable par variable
# ---------------------------------------------------------------------

for column in rare_missing_columns:

    # -------------------------------------------------------------
    # Nombre de NaN avant traitement
    # -------------------------------------------------------------

    train_missing_count = int(
        X_train_processed[column]
        .isna()
        .sum()
    )

    test_missing_count = int(
        X_test_processed[column]
        .isna()
        .sum()
    )


    # -------------------------------------------------------------
    # Médiane calculée exclusivement sur X_train_processed
    # -------------------------------------------------------------

    median_train = (
        X_train_processed[column]
        .median()
    )

    if pd.isna(median_train):
        raise ValueError(
            f"Impossible de calculer la médiane de '{column}' "
            "à partir de X_train_processed."
        )


    # -------------------------------------------------------------
    # Imputation avec la médiane apprise sur Train
    # -------------------------------------------------------------

    X_train_processed[column] = (
        X_train_processed[column]
        .fillna(median_train)
    )

    X_test_processed[column] = (
        X_test_processed[column]
        .fillna(median_train)
    )


    # -------------------------------------------------------------
    # Contrôles après traitement
    # -------------------------------------------------------------

    train_remaining_nan = int(
        X_train_processed[column]
        .isna()
        .sum()
    )

    test_remaining_nan = int(
        X_test_processed[column]
        .isna()
        .sum()
    )

    if train_remaining_nan != 0:
        raise ValueError(
            f"Des NaN subsistent dans '{column}' "
            "pour X_train_processed."
        )

    if test_remaining_nan != 0:
        raise ValueError(
            f"Des NaN subsistent dans '{column}' "
            "pour X_test_processed."
        )


    # -------------------------------------------------------------
    # Rapport
    # -------------------------------------------------------------

    rare_imputation_report.extend(
        [
            {
                "variable": column,
                "dataset": "Train",
                "nan_avant_traitement": train_missing_count,
                "median_train": median_train,
                "nan_apres_traitement": train_remaining_nan,
            },
            {
                "variable": column,
                "dataset": "Test",
                "nan_avant_traitement": test_missing_count,
                "median_train": median_train,
                "nan_apres_traitement": test_remaining_nan,
            },
        ]
    )


# ---------------------------------------------------------------------
# Construction du rapport
# ---------------------------------------------------------------------

rare_imputation_report_df = pd.DataFrame(
    rare_imputation_report
)


# ---------------------------------------------------------------------
# Résumé contextualisé
# ---------------------------------------------------------------------

print(
    "=== Imputation des valeurs manquantes rares ==="
)

print(
    f"\nNombre de variables concernées : "
    f"{len(rare_missing_columns)}"
)

print(
    "\nVariables imputées :"
)

for column in rare_missing_columns:
    print(f"  - {column}")

print(
    "\nPour chaque variable, la médiane a été calculée "
    "exclusivement sur X_train_processed puis appliquée "
    "à X_train_processed et X_test_processed."
)

print(
    "\nAucune statistique n'a été calculée à partir "
    "du jeu de test."
)

print(
    "\nContrôle : aucun NaN ne subsiste dans les "
    "variables traitées."
)


# ---------------------------------------------------------------------
# Affichage du rapport
# ---------------------------------------------------------------------

display(
    rare_imputation_report_df
)

=== Imputation des valeurs manquantes rares ===

Nombre de variables concernées : 2

Variables imputées :
  - wltp_test_mass_kg
  - engine_power_kw

Pour chaque variable, la médiane a été calculée exclusivement sur X_train_processed puis appliquée à X_train_processed et X_test_processed.

Aucune statistique n'a été calculée à partir du jeu de test.

Contrôle : aucun NaN ne subsiste dans les variables traitées.


,variable,dataset,nan_avant_traitement,median_train,nan_apres_traitement
0,wltp_test_mass_kg,Train,225,1609.0,0
1,wltp_test_mass_kg,Test,60,1609.0,0
2,engine_power_kw,Train,1,103.0,0
3,engine_power_kw,Test,0,103.0,0


### 10.9 Synthèse des traitements numériques appliqués

Les traitements définis au point 10.4 ont été appliqués aux étapes 10.6,
10.7 et 10.8.

Cette étape n'effectue aucune nouvelle imputation et ne modifie pas
`X_train_processed` ou `X_test_processed`.

Elle regroupe uniquement les résultats des traitements précédents dans un
tableau de synthèse permettant d'identifier, pour chaque variable :

- le jeu concerné (`Train` ou `Test`) ;
- la nature des valeurs manquantes ;
- le nombre de valeurs manquantes traitées ;
- le traitement réellement appliqué ;
- la valeur utilisée pour le remplacement ;
- l'origine de cette valeur.

Pour les imputations statistiques, la valeur utilisée est toujours une
médiane apprise exclusivement sur `X_train`.

Pour les absences structurelles ou informatives traitées par `0`, la valeur
provient de la règle métier définie au point 10.4.

In [19]:
# ---------------------------------------------------------------------
# 10.9 - Synthèse des traitements numériques appliqués
# ---------------------------------------------------------------------

imputation_summary = []


# ---------------------------------------------------------------------
# 1. Traitements conditionnels réalisés au point 10.6
# ---------------------------------------------------------------------

for _, row in conditional_imputation_report_df.iterrows():

    # NaN structurels - Train
    if row["train_structural_nan"] > 0:
        imputation_summary.append(
            {
                "variable": row["variable"],
                "dataset": "Train",
                "nature_nan": "Structurel",
                "nombre_nan_traites": int(
                    row["train_structural_nan"]
                ),
                "traitement": "Remplacement par 0",
                "valeur_utilisee": 0.0,
                "origine_valeur": "Règle métier",
            }
        )

    # NaN résiduels - Train
    if row["train_residual_nan"] > 0:
        imputation_summary.append(
            {
                "variable": row["variable"],
                "dataset": "Train",
                "nature_nan": "Résiduel",
                "nombre_nan_traites": int(
                    row["train_residual_nan"]
                ),
                "traitement": "Imputation par médiane",
                "valeur_utilisee": row["median_train"],
                "origine_valeur": "Médiane apprise sur X_train",
            }
        )

    # NaN structurels - Test
    if row["test_structural_nan"] > 0:
        imputation_summary.append(
            {
                "variable": row["variable"],
                "dataset": "Test",
                "nature_nan": "Structurel",
                "nombre_nan_traites": int(
                    row["test_structural_nan"]
                ),
                "traitement": "Remplacement par 0",
                "valeur_utilisee": 0.0,
                "origine_valeur": "Règle métier",
            }
        )

    # NaN résiduels - Test
    if row["test_residual_nan"] > 0:
        imputation_summary.append(
            {
                "variable": row["variable"],
                "dataset": "Test",
                "nature_nan": "Résiduel",
                "nombre_nan_traites": int(
                    row["test_residual_nan"]
                ),
                "traitement": "Imputation par médiane",
                "valeur_utilisee": row["median_train"],
                "origine_valeur": "Médiane apprise sur X_train",
            }
        )


# ---------------------------------------------------------------------
# 2. Traitement spécifique réalisé au point 10.7
# ---------------------------------------------------------------------

for _, row in co2_reduction_report_df.iterrows():

    if row["nan_avant_traitement"] > 0:
        imputation_summary.append(
            {
                "variable": "co2_reduction_wltp_g_km",
                "dataset": row["dataset"],
                "nature_nan": "Absence informative",
                "nombre_nan_traites": int(
                    row["nan_avant_traitement"]
                ),
                "traitement": "Remplacement par 0",
                "valeur_utilisee": 0.0,
                "origine_valeur": "Règle métier",
            }
        )


# ---------------------------------------------------------------------
# 3. Imputations des NaN rares réalisées au point 10.8
# ---------------------------------------------------------------------

for _, row in rare_imputation_report_df.iterrows():

    # Une ligne n'est ajoutée que lorsqu'une imputation
    # a réellement été nécessaire.
    if row["nan_avant_traitement"] > 0:
        imputation_summary.append(
            {
                "variable": row["variable"],
                "dataset": row["dataset"],
                "nature_nan": "Résiduel rare",
                "nombre_nan_traites": int(
                    row["nan_avant_traitement"]
                ),
                "traitement": "Imputation par médiane",
                "valeur_utilisee": row["median_train"],
                "origine_valeur": "Médiane apprise sur X_train",
            }
        )


# ---------------------------------------------------------------------
# 4. Construction du tableau de synthèse
# ---------------------------------------------------------------------

imputation_summary_df = pd.DataFrame(
    imputation_summary
)


# Ordre d'affichage des variables.
variable_order = [
    "engine_capacity_cm3",
    "electric_energy_consumption_wh_km",
    "electric_range_km",
    "fuel_consumption",
    "co2_reduction_wltp_g_km",
    "wltp_test_mass_kg",
    "engine_power_kw",
]

imputation_summary_df["variable"] = pd.Categorical(
    imputation_summary_df["variable"],
    categories=variable_order,
    ordered=True,
)


# Train affiché avant Test.
imputation_summary_df["dataset"] = pd.Categorical(
    imputation_summary_df["dataset"],
    categories=["Train", "Test"],
    ordered=True,
)


imputation_summary_df = (
    imputation_summary_df
    .sort_values(
        by=[
            "variable",
            "dataset",
            "nature_nan",
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# 5. Affichage
# ---------------------------------------------------------------------

display(imputation_summary_df)

,variable,dataset,nature_nan,nombre_nan_traites,traitement,valeur_utilisee,origine_valeur
0,engine_capacity_cm3,Train,Structurel,11355,Remplacement par 0,0.0,Règle métier
1,engine_capacity_cm3,Test,Structurel,2840,Remplacement par 0,0.0,Règle métier
2,electric_energy_consumption_wh_km,Train,Résiduel,133,Imputation par médiane,167.0,Médiane apprise sur X_train
3,electric_energy_consumption_wh_km,Train,Structurel,63063,Remplacement par 0,0.0,Règle métier
4,electric_energy_consumption_wh_km,Test,Résiduel,33,Imputation par médiane,167.0,Médiane apprise sur X_train
5,electric_energy_consumption_wh_km,Test,Structurel,15733,Remplacement par 0,0.0,Règle métier
6,electric_range_km,Train,Résiduel,152,Imputation par médiane,394.0,Médiane apprise sur X_train
7,electric_range_km,Train,Structurel,63063,Remplacement par 0,0.0,Règle métier
8,electric_range_km,Test,Résiduel,39,Imputation par médiane,394.0,Médiane apprise sur X_train
9,electric_range_km,Test,Structurel,15733,Remplacement par 0,0.0,Règle métier


### 10.10 Validation finale des traitements appliqués

Les traitements définis au point 10.4 et appliqués aux points 10.5 à 10.8
sont maintenant contrôlés avant de poursuivre le preprocessing.

Cette étape n'effectue aucune nouvelle transformation.

Les vérifications portent sur :

1. **Les variables numériques traitées**

   Les variables numériques ayant fait l'objet d'un traitement ne doivent
   plus contenir de valeur manquante.

2. **Les indicateurs binaires**

   Les quatre indicateurs créés au point 10.5 doivent :

   - être présents dans `X_train_processed` et `X_test_processed` ;
   - ne contenir aucune valeur manquante ;
   - contenir uniquement les valeurs `0` et `1`.

3. **Le contrôle global des valeurs manquantes**

   Après application de l'ensemble des traitements définis précédemment,
   aucune valeur manquante ne doit subsister dans `X_train_processed`
   ou `X_test_processed`.

L'objectif est de valider définitivement le traitement des valeurs manquantes
avant de poursuivre avec l'encodage des variables catégorielles et les étapes
suivantes du preprocessing.

In [20]:
# ---------------------------------------------------------------------
# 10.10 - Validation finale des traitements appliqués
# ---------------------------------------------------------------------

treated_numeric_columns = [
    "engine_capacity_cm3",
    "electric_energy_consumption_wh_km",
    "electric_range_km",
    "fuel_consumption",
    "co2_reduction_wltp_g_km",
    "wltp_test_mass_kg",
    "engine_power_kw",
]

indicator_columns = [
    "has_electric_energy_consumption_wh_km",
    "has_electric_range_km",
    "has_fuel_consumption",
    "has_co2_reduction_wltp_g_km",
]


# ---------------------------------------------------------------------
# 1. Vérification des variables numériques traitées
# ---------------------------------------------------------------------

numeric_validation = []

for column in treated_numeric_columns:

    train_nan = int(
        X_train_processed[column]
        .isna()
        .sum()
    )

    test_nan = int(
        X_test_processed[column]
        .isna()
        .sum()
    )

    numeric_validation.append(
        {
            "variable": column,
            "nan_train": train_nan,
            "nan_test": test_nan,
            "statut": (
                "OK"
                if train_nan == 0 and test_nan == 0
                else "À vérifier"
            ),
        }
    )


numeric_validation_df = pd.DataFrame(
    numeric_validation
)

print(
    "=== Validation finale des traitements "
    "des valeurs manquantes ==="
)

print(
    "\n1. Variables numériques traitées"
)

print(
    "Les variables suivantes ont fait l'objet d'un traitement "
    "aux étapes précédentes. Aucune valeur manquante ne doit "
    "désormais subsister."
)

display(numeric_validation_df)


# ---------------------------------------------------------------------
# 2. Vérification des indicateurs binaires
# ---------------------------------------------------------------------

indicator_validation = []

for column in indicator_columns:

    train_exists = (
        column in X_train_processed.columns
    )

    test_exists = (
        column in X_test_processed.columns
    )

    if train_exists and test_exists:

        train_nan = int(
            X_train_processed[column]
            .isna()
            .sum()
        )

        test_nan = int(
            X_test_processed[column]
            .isna()
            .sum()
        )

        train_values = set(
            X_train_processed[column]
            .dropna()
            .unique()
        )

        test_values = set(
            X_test_processed[column]
            .dropna()
            .unique()
        )

        binary_values_valid = (
            train_values.issubset({0, 1})
            and test_values.issubset({0, 1})
        )

        valid = (
            train_nan == 0
            and test_nan == 0
            and binary_values_valid
        )

    else:

        train_nan = None
        test_nan = None
        binary_values_valid = False
        valid = False


    indicator_validation.append(
        {
            "indicateur": column,
            "present_train": train_exists,
            "present_test": test_exists,
            "nan_train": train_nan,
            "nan_test": test_nan,
            "valeurs_binaires_valides":
                binary_values_valid,
            "statut":
                "OK" if valid else "À vérifier",
        }
    )


indicator_validation_df = pd.DataFrame(
    indicator_validation
)

print(
    "\n2. Indicateurs binaires"
)

print(
    "Les quatre indicateurs doivent être présents dans "
    "Train et Test, sans NaN et avec uniquement "
    "les valeurs 0 et 1."
)

display(indicator_validation_df)


# ---------------------------------------------------------------------
# 3. Contrôle global des valeurs manquantes
# ---------------------------------------------------------------------

remaining_train_nan = (
    X_train_processed
    .isna()
    .sum()
)

remaining_train_nan = (
    remaining_train_nan[
        remaining_train_nan > 0
    ]
)


remaining_test_nan = (
    X_test_processed
    .isna()
    .sum()
)

remaining_test_nan = (
    remaining_test_nan[
        remaining_test_nan > 0
    ]
)


remaining_nan_columns = sorted(
    set(remaining_train_nan.index)
    | set(remaining_test_nan.index)
)


remaining_nan_report = []

for column in remaining_nan_columns:

    remaining_nan_report.append(
        {
            "variable": column,
            "nan_train": int(
                remaining_train_nan.get(
                    column,
                    0,
                )
            ),
            "nan_test": int(
                remaining_test_nan.get(
                    column,
                    0,
                )
            ),
        }
    )


remaining_nan_report_df = pd.DataFrame(
    remaining_nan_report
)


print(
    "\n3. Contrôle global des valeurs manquantes"
)

if remaining_nan_report_df.empty:

    print(
        "Aucune valeur manquante ne subsiste dans "
        "X_train_processed ou X_test_processed."
    )

else:

    print(
        "Des valeurs manquantes subsistent encore "
        "dans les variables suivantes :"
    )

    display(
        remaining_nan_report_df
    )


# ---------------------------------------------------------------------
# 4. Validation finale
# ---------------------------------------------------------------------

numeric_treatment_valid = (
    numeric_validation_df["statut"]
    .eq("OK")
    .all()
)

indicators_valid = (
    indicator_validation_df["statut"]
    .eq("OK")
    .all()
)

no_remaining_nan = (
    remaining_nan_report_df.empty
)


if (
    numeric_treatment_valid
    and indicators_valid
    and no_remaining_nan
):

    print(
        "\n✅ Validation finale réussie : "
        "aucune valeur manquante ne subsiste et "
        "les indicateurs binaires sont conformes."
    )

else:

    raise ValueError(
        "La validation finale du traitement "
        "des valeurs manquantes a échoué."
    )

=== Validation finale des traitements des valeurs manquantes ===

1. Variables numériques traitées
Les variables suivantes ont fait l'objet d'un traitement aux étapes précédentes. Aucune valeur manquante ne doit désormais subsister.


,variable,nan_train,nan_test,statut
0,engine_capacity_cm3,0,0,OK
1,electric_energy_consumption_wh_km,0,0,OK
2,electric_range_km,0,0,OK
3,fuel_consumption,0,0,OK
4,co2_reduction_wltp_g_km,0,0,OK
5,wltp_test_mass_kg,0,0,OK
6,engine_power_kw,0,0,OK



2. Indicateurs binaires
Les quatre indicateurs doivent être présents dans Train et Test, sans NaN et avec uniquement les valeurs 0 et 1.


,indicateur,present_train,present_test,nan_train,nan_test,valeurs_binaires_valides,statut
0,has_electric_energy_consumption_wh_km,True,True,0,0,True,OK
1,has_electric_range_km,True,True,0,0,True,OK
2,has_fuel_consumption,True,True,0,0,True,OK
3,has_co2_reduction_wltp_g_km,True,True,0,0,True,OK



3. Contrôle global des valeurs manquantes
Aucune valeur manquante ne subsiste dans X_train_processed ou X_test_processed.

✅ Validation finale réussie : aucune valeur manquante ne subsiste et les indicateurs binaires sont conformes.


### 10.11 Validation globale des valeurs manquantes

Les traitements des valeurs manquantes numériques et catégorielles ayant été
appliqués, une vérification globale est réalisée avant de poursuivre le
preprocessing.

Cette étape n'effectue aucune transformation supplémentaire.

Elle vérifie que :

- `X_train_processed` ne contient plus aucune valeur manquante ;
- `X_test_processed` ne contient plus aucune valeur manquante.

Cette validation clôture le traitement des valeurs manquantes avant les
étapes suivantes d'encodage des variables catégorielles et de mise à
l'échelle des variables numériques.

In [21]:
# ---------------------------------------------------------------------
# 10.11 - Validation globale des valeurs manquantes
# ---------------------------------------------------------------------

# Nombre total de NaN restants.
train_total_nan = int(
    X_train_processed
    .isna()
    .sum()
    .sum()
)

test_total_nan = int(
    X_test_processed
    .isna()
    .sum()
    .sum()
)


# ---------------------------------------------------------------------
# Construction du rapport
# ---------------------------------------------------------------------

missing_validation_df = pd.DataFrame(
    [
        {
            "dataset": "Train",
            "nombre_total_nan": train_total_nan,
            "statut": (
                "OK"
                if train_total_nan == 0
                else "À vérifier"
            ),
        },
        {
            "dataset": "Test",
            "nombre_total_nan": test_total_nan,
            "statut": (
                "OK"
                if test_total_nan == 0
                else "À vérifier"
            ),
        },
    ]
)


# ---------------------------------------------------------------------
# Résumé contextualisé
# ---------------------------------------------------------------------

print(
    "=== Validation globale des valeurs manquantes ==="
)

print(
    "\nCette étape vérifie l'ensemble des variables présentes "
    "dans X_train_processed et X_test_processed."
)

print(
    f"\nNombre de variables dans X_train_processed : "
    f"{X_train_processed.shape[1]}"
)

print(
    f"Nombre de variables dans X_test_processed  : "
    f"{X_test_processed.shape[1]}"
)

print(
    "\nAucune valeur manquante ne doit subsister "
    "avant l'étape d'encodage."
)


# ---------------------------------------------------------------------
# Affichage du contrôle
# ---------------------------------------------------------------------

display(
    missing_validation_df
)


# ---------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------

if train_total_nan != 0 or test_total_nan != 0:

    raise ValueError(
        "Des valeurs manquantes subsistent dans "
        "X_train_processed ou X_test_processed."
    )


print(
    "\n✅ Validation globale réussie : "
    "aucun NaN ne subsiste dans "
    "X_train_processed ni dans X_test_processed."
)

=== Validation globale des valeurs manquantes ===

Cette étape vérifie l'ensemble des variables présentes dans X_train_processed et X_test_processed.

Nombre de variables dans X_train_processed : 17
Nombre de variables dans X_test_processed  : 17

Aucune valeur manquante ne doit subsister avant l'étape d'encodage.


,dataset,nombre_total_nan,statut
0,Train,0,OK
1,Test,0,OK



✅ Validation globale réussie : aucun NaN ne subsiste dans X_train_processed ni dans X_test_processed.


### 10.12 Stratégie d'encodage des variables catégorielles nominales

Les valeurs manquantes ayant été entièrement traitées et validées, l'étape
suivante consiste à préparer l'encodage des variables catégorielles nominales.

Les trois variables catégorielles retenues pour la modélisation sont :

- `vehicle_category_type` ;
- `fuel_type` ;
- `fuel_mode`.

L'analyse réalisée précédemment a montré qu'elles sont toutes **nominales** et
présentent une faible cardinalité.

La stratégie retenue est donc un **One-Hot Encoding** pour ces trois variables.

Chaque modalité sera représentée par une variable binaire indépendante.

L'encodeur sera :

1. ajusté exclusivement sur `X_train_processed` ;
2. utilisé ensuite pour transformer `X_train_processed` ;
3. réutilisé sans nouvel apprentissage pour transformer `X_test_processed`.

Afin de rendre le preprocessing robuste à de nouvelles données, l'encodeur
devra également pouvoir gérer une éventuelle modalité absente du jeu
d'entraînement sans provoquer d'erreur.

Avant l'encodage, un contrôle est réalisé afin de vérifier :

- la présence des trois variables dans Train et Test ;
- leur cardinalité respective ;
- l'existence éventuelle de modalités présentes dans Test mais absentes de Train.

Aucune cardinalité n'est considérée comme fixe : les résultats sont calculés
à partir des données effectivement présentes lors de l'exécution.

In [22]:
# ---------------------------------------------------------------------
# 10.12 - Contrôle préalable à l'encodage des variables catégorielles
# ---------------------------------------------------------------------

categorical_columns = [
    "vehicle_category_type",
    "fuel_type",
    "fuel_mode",
]


# ---------------------------------------------------------------------
# 1. Contrôle de présence des variables
# ---------------------------------------------------------------------

missing_train_columns = [
    column
    for column in categorical_columns
    if column not in X_train_processed.columns
]

missing_test_columns = [
    column
    for column in categorical_columns
    if column not in X_test_processed.columns
]

if missing_train_columns or missing_test_columns:
    raise ValueError(
        "Certaines variables catégorielles sont absentes des datasets.\n"
        f"Absentes de X_train_processed : {missing_train_columns}\n"
        f"Absentes de X_test_processed : {missing_test_columns}"
    )


# ---------------------------------------------------------------------
# 2. Cardinalité observée dans Train et Test
# ---------------------------------------------------------------------

categorical_cardinality_report = []

for column in categorical_columns:

    train_categories = set(
        X_train_processed[column]
        .dropna()
        .unique()
    )

    test_categories = set(
        X_test_processed[column]
        .dropna()
        .unique()
    )

    unseen_test_categories = (
        test_categories - train_categories
    )

    categorical_cardinality_report.append(
        {
            "variable": column,
            "modalites_train": len(train_categories),
            "modalites_test": len(test_categories),
            "modalites_test_absentes_train": len(
                unseen_test_categories
            ),
        }
    )


categorical_cardinality_df = pd.DataFrame(
    categorical_cardinality_report
)


# ---------------------------------------------------------------------
# 3. Identification des modalités présentes dans Test
#    mais absentes de Train
# ---------------------------------------------------------------------

unseen_categories_report = []

for column in categorical_columns:

    train_categories = set(
        X_train_processed[column]
        .dropna()
        .unique()
    )

    test_categories = set(
        X_test_processed[column]
        .dropna()
        .unique()
    )

    unseen_categories = sorted(
        test_categories - train_categories
    )

    for category in unseen_categories:

        unseen_categories_report.append(
            {
                "variable": column,
                "modalite_absente_train": category,
            }
        )


unseen_categories_df = pd.DataFrame(
    unseen_categories_report
)


# ---------------------------------------------------------------------
# 4. Résumé contextualisé
# ---------------------------------------------------------------------

print(
    "=== Contrôle préalable au One-Hot Encoding ==="
)

print(
    f"\nNombre de variables catégorielles retenues : "
    f"{len(categorical_columns)}"
)

print(
    "\nVariables concernées :"
)

for column in categorical_columns:
    print(f"  - {column}")

print(
    "\nLe tableau suivant compare la cardinalité observée "
    "dans X_train_processed et X_test_processed."
)

display(
    categorical_cardinality_df
)


# ---------------------------------------------------------------------
# 5. Modalités inconnues dans Test
# ---------------------------------------------------------------------

print(
    "\n=== Modalités présentes dans Test "
    "mais absentes de Train ==="
)

if unseen_categories_df.empty:

    print(
        "\nAucune modalité de X_test_processed "
        "n'est absente de X_train_processed "
        "pour les trois variables catégorielles."
    )

else:

    print(
        "\nLes modalités suivantes sont présentes dans Test "
        "mais absentes de Train :"
    )

    display(
        unseen_categories_df
    )


# ---------------------------------------------------------------------
# 6. Conclusion
# ---------------------------------------------------------------------

print(
    "\n✅ Contrôle préalable terminé : "
    "les trois variables catégorielles sont prêtes "
    "pour le One-Hot Encoding."
)

=== Contrôle préalable au One-Hot Encoding ===

Nombre de variables catégorielles retenues : 3

Variables concernées :
  - vehicle_category_type
  - fuel_type
  - fuel_mode

Le tableau suivant compare la cardinalité observée dans X_train_processed et X_test_processed.


,variable,modalites_train,modalites_test,modalites_test_absentes_train
0,vehicle_category_type,2,1,0
1,fuel_type,9,9,0
2,fuel_mode,6,6,0



=== Modalités présentes dans Test mais absentes de Train ===

Aucune modalité de X_test_processed n'est absente de X_train_processed pour les trois variables catégorielles.

✅ Contrôle préalable terminé : les trois variables catégorielles sont prêtes pour le One-Hot Encoding.


### 10.13 One-Hot Encoding des variables catégorielles à faible cardinalité

Conformément à la stratégie définie au point 10.12, les trois variables
catégorielles nominales retenues pour la modélisation sont traitées par
**One-Hot Encoding** :

- `vehicle_category_type` ;
- `fuel_type` ;
- `fuel_mode`.

L'encodeur est ajusté exclusivement sur `X_train_processed`.

Les catégories apprises à partir du jeu d'entraînement sont ensuite utilisées
pour transformer :

- `X_train_processed` ;
- `X_test_processed`.

La gestion d'éventuelles catégories inconnues est assurée par
`handle_unknown="ignore"` afin qu'une modalité absente du jeu d'entraînement
ne provoque pas d'erreur lors de la transformation du jeu de test ou de
futures données.

Les trois variables catégorielles originales sont ensuite supprimées et
remplacées par les variables binaires produites par le One-Hot Encoding.

Enfin, un contrôle vérifie que `X_train_processed` et `X_test_processed`
possèdent exactement les mêmes variables après l'encodage.

In [23]:
# ---------------------------------------------------------------------
# 10.13 - One-Hot Encoding des variables catégorielles
#         à faible cardinalité
# ---------------------------------------------------------------------

from sklearn.preprocessing import OneHotEncoder


# Variables concernées uniquement par le One-Hot Encoding.
onehot_columns = [
    "vehicle_category_type",
    "fuel_type",
    "fuel_mode",
]


# ---------------------------------------------------------------------
# 1. Création et apprentissage de l'encodeur sur Train uniquement
# ---------------------------------------------------------------------

onehot_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
    dtype="int8",
)

onehot_encoder.fit(
    X_train_processed[onehot_columns]
)


# ---------------------------------------------------------------------
# 2. Transformation de Train et Test
# ---------------------------------------------------------------------

X_train_onehot_array = onehot_encoder.transform(
    X_train_processed[onehot_columns]
)

X_test_onehot_array = onehot_encoder.transform(
    X_test_processed[onehot_columns]
)


# ---------------------------------------------------------------------
# 3. Récupération dynamique des noms des nouvelles variables
# ---------------------------------------------------------------------

onehot_feature_names = (
    onehot_encoder
    .get_feature_names_out(onehot_columns)
)


# ---------------------------------------------------------------------
# 4. Conversion des résultats en DataFrames
# ---------------------------------------------------------------------

X_train_onehot = pd.DataFrame(
    X_train_onehot_array,
    columns=onehot_feature_names,
    index=X_train_processed.index,
)

X_test_onehot = pd.DataFrame(
    X_test_onehot_array,
    columns=onehot_feature_names,
    index=X_test_processed.index,
)


# ---------------------------------------------------------------------
# 5. Suppression des variables catégorielles originales
# ---------------------------------------------------------------------

X_train_processed.drop(
    columns=onehot_columns,
    inplace=True,
)

X_test_processed.drop(
    columns=onehot_columns,
    inplace=True,
)


# ---------------------------------------------------------------------
# 6. Ajout des variables One-Hot encodées
# ---------------------------------------------------------------------

X_train_processed = pd.concat(
    [
        X_train_processed,
        X_train_onehot,
    ],
    axis=1,
)

X_test_processed = pd.concat(
    [
        X_test_processed,
        X_test_onehot,
    ],
    axis=1,
)


# ---------------------------------------------------------------------
# 7. Vérification de la cohérence des colonnes Train / Test
# ---------------------------------------------------------------------

same_columns = (
    X_train_processed.columns.tolist()
    == X_test_processed.columns.tolist()
)

if not same_columns:
    raise ValueError(
        "Les colonnes de X_train_processed et X_test_processed "
        "ne sont pas identiques après One-Hot Encoding."
    )


# ---------------------------------------------------------------------
# 8. Rapport du traitement
# ---------------------------------------------------------------------

onehot_report_df = pd.DataFrame(
    [
        {
            "variable_source": column,
            "modalites_apprises_train": len(
                onehot_encoder.categories_[index]
            ),
        }
        for index, column in enumerate(onehot_columns)
    ]
)

display(onehot_report_df)

print(
    f"\nNombre total de variables créées par One-Hot Encoding : "
    f"{len(onehot_feature_names)}"
)

print(
    "\n✅ One-Hot Encoding terminé : "
    "Train et Test possèdent exactement les mêmes colonnes."
)

,variable_source,modalites_apprises_train
0,vehicle_category_type,2
1,fuel_type,9
2,fuel_mode,6



Nombre total de variables créées par One-Hot Encoding : 17

✅ One-Hot Encoding terminé : Train et Test possèdent exactement les mêmes colonnes.


### 10.14 Standardisation des variables numériques continues

Après le traitement des valeurs manquantes et l'encodage des variables
catégorielles, les variables numériques continues sont mises à l'échelle
à l'aide d'un **StandardScaler**.

La standardisation transforme chaque variable à partir de la moyenne et de
l'écart-type appris sur les données d'entraînement.

Afin d'éviter toute fuite d'information :

- le `StandardScaler` est ajusté exclusivement sur `X_train_processed` ;
- les paramètres appris sur Train sont utilisés pour transformer
  `X_train_processed` ;
- ces mêmes paramètres sont ensuite utilisés pour transformer
  `X_test_processed`.

Les variables binaires ne sont pas standardisées :

- les quatre indicateurs `has_*` créés lors du traitement des valeurs manquantes ;
- les variables binaires produites par le One-Hot Encoding.

Elles conservent donc directement leurs valeurs `0` et `1`.

Les autres variables numériques sont considérées comme les variables à
standardiser.

La liste des variables concernées est déterminée dynamiquement à partir de
`X_train_processed`, puis un contrôle vérifie que ces mêmes variables sont
présentes dans `X_test_processed`.

In [ ]:
# ---------------------------------------------------------------------
# 10.14 - Standardisation des variables numériques continues
# ---------------------------------------------------------------------

from sklearn.preprocessing import StandardScaler


# ---------------------------------------------------------------------
# 1. Identification des variables binaires à ne pas standardiser
# ---------------------------------------------------------------------

binary_indicator_columns = [
    "has_electric_energy_consumption_wh_km",
    "has_electric_range_km",
    "has_fuel_consumption",
    "has_co2_reduction_wltp_g_km",
]

# Variables créées par le One-Hot Encoding au point 10.13.
onehot_encoded_columns = list(
    onehot_feature_names
)

excluded_from_scaling = (
    binary_indicator_columns
    + onehot_encoded_columns
)


# ---------------------------------------------------------------------
# 2. Identification dynamique des variables numériques continues
# ---------------------------------------------------------------------

numeric_columns = (
    X_train_processed
    .select_dtypes(include="number")
    .columns
    .tolist()
)

columns_to_scale = [
    column
    for column in numeric_columns
    if column not in excluded_from_scaling
]


# ---------------------------------------------------------------------
# 3. Contrôle de cohérence Train / Test
# ---------------------------------------------------------------------

missing_in_test = [
    column
    for column in columns_to_scale
    if column not in X_test_processed.columns
]

if missing_in_test:
    raise ValueError(
        "Certaines variables numériques de Train sont absentes de Test : "
        f"{missing_in_test}"
    )


# ---------------------------------------------------------------------
# 4. Apprentissage du StandardScaler exclusivement sur Train
# ---------------------------------------------------------------------

standard_scaler = StandardScaler()

standard_scaler.fit(
    X_train_processed[columns_to_scale]
)


# ---------------------------------------------------------------------
# 5. Transformation de Train et Test
# ---------------------------------------------------------------------

X_train_processed[columns_to_scale] = (
    standard_scaler.transform(
        X_train_processed[columns_to_scale]
    )
)

X_test_processed[columns_to_scale] = (
    standard_scaler.transform(
        X_test_processed[columns_to_scale]
    )
)


# ---------------------------------------------------------------------
# 6. Rapport de standardisation
# ---------------------------------------------------------------------

scaling_report_df = pd.DataFrame(
    {
        "variable": columns_to_scale,
        "moyenne_apprise_train": standard_scaler.mean_,
        "ecart_type_appris_train": standard_scaler.scale_,
    }
)

display(scaling_report_df)


print(
    f"\nNombre de variables standardisées : "
    f"{len(columns_to_scale)}"
)

print(
    f"Nombre de variables binaires exclues du scaling : "
    f"{len(excluded_from_scaling)}"
)

print(
    "\n✅ StandardScaler appris exclusivement sur Train "
    "et appliqué à Train et Test."
)

,variable,moyenne_apprise_train,ecart_type_appris_train
0,mass_running_order_kg,1566.452712,359.187662
1,wltp_test_mass_kg,1686.629600,379.390468
2,engine_capacity_cm3,1352.332175,747.314823
3,engine_power_kw,117.807812,62.547350
4,electric_energy_consumption_wh_km,37.027475,73.087651
5,co2_reduction_wltp_g_km,0.842596,0.844135
6,fuel_consumption,4.717263,2.524487
7,electric_range_km,69.009563,160.329387
8,registration_month_sin,0.036490,0.711809
9,registration_month_cos,0.006502,0.701395



Nombre de variables standardisées : 10
Nombre de variables binaires exclues du scaling : 21

✅ StandardScaler appris exclusivement sur Train et appliqué à Train et Test.


### 10.15 Validation finale du preprocessing

Les principales transformations du preprocessing ont maintenant été appliquées
à `X_train_processed` et `X_test_processed`.

Cette étape réalise une validation finale avant l'export des données préparées
pour la modélisation.

Les contrôles portent sur :

1. **La structure des jeux de données**
   - même nombre de variables dans Train et Test ;
   - mêmes noms de colonnes ;
   - même ordre des colonnes.

2. **Les valeurs manquantes**
   - aucun `NaN` ne doit subsister dans les variables explicatives.

3. **Les variables catégorielles originales**
   - `vehicle_category_type` ;
   - `fuel_type` ;
   - `fuel_mode`.

   Ces variables doivent avoir disparu après le One-Hot Encoding.

4. **Les types de données**
   - toutes les variables finales doivent être numériques.

5. **Les indicateurs binaires**
   - les quatre indicateurs `has_*` doivent être présents ;
   - ils doivent conserver uniquement les valeurs `0` et `1`.

6. **Les dimensions finales**

   Les dimensions de `X_train_processed` et `X_test_processed` sont affichées
   afin de documenter le dataset obtenu après preprocessing.

Cette validation n'effectue aucune nouvelle transformation. Elle vérifie
uniquement la conformité des jeux de données avant leur utilisation pour
l'entraînement et l'évaluation des modèles.

In [25]:
# ---------------------------------------------------------------------
# 10.15 - Validation finale du preprocessing
# ---------------------------------------------------------------------

original_categorical_columns = [
    "vehicle_category_type",
    "fuel_type",
    "fuel_mode",
]

indicator_columns = [
    "has_electric_energy_consumption_wh_km",
    "has_electric_range_km",
    "has_fuel_consumption",
    "has_co2_reduction_wltp_g_km",
]


# ---------------------------------------------------------------------
# 1. Structure Train / Test
# ---------------------------------------------------------------------

same_number_columns = (
    X_train_processed.shape[1]
    == X_test_processed.shape[1]
)

same_column_names = (
    set(X_train_processed.columns)
    == set(X_test_processed.columns)
)

same_column_order = (
    list(X_train_processed.columns)
    == list(X_test_processed.columns)
)


# ---------------------------------------------------------------------
# 2. Valeurs manquantes
# ---------------------------------------------------------------------

train_total_nan = int(
    X_train_processed
    .isna()
    .sum()
    .sum()
)

test_total_nan = int(
    X_test_processed
    .isna()
    .sum()
    .sum()
)

no_missing_values = (
    train_total_nan == 0
    and test_total_nan == 0
)


# ---------------------------------------------------------------------
# 3. Disparition des variables catégorielles originales
# ---------------------------------------------------------------------

remaining_original_categorical = [
    column
    for column in original_categorical_columns
    if (
        column in X_train_processed.columns
        or column in X_test_processed.columns
    )
]

categorical_encoding_valid = (
    len(remaining_original_categorical) == 0
)


# ---------------------------------------------------------------------
# 4. Vérification des types de données
# ---------------------------------------------------------------------

train_non_numeric_columns = (
    X_train_processed
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)

test_non_numeric_columns = (
    X_test_processed
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)

all_columns_numeric = (
    len(train_non_numeric_columns) == 0
    and len(test_non_numeric_columns) == 0
)


# ---------------------------------------------------------------------
# 5. Vérification des indicateurs binaires
# ---------------------------------------------------------------------

indicators_valid = True

indicator_report = []

for column in indicator_columns:

    train_exists = (
        column in X_train_processed.columns
    )

    test_exists = (
        column in X_test_processed.columns
    )

    if train_exists and test_exists:

        train_values = set(
            X_train_processed[column]
            .unique()
        )

        test_values = set(
            X_test_processed[column]
            .unique()
        )

        binary_values_valid = (
            train_values.issubset({0, 1})
            and test_values.issubset({0, 1})
        )

    else:

        binary_values_valid = False

    valid = (
        train_exists
        and test_exists
        and binary_values_valid
    )

    if not valid:
        indicators_valid = False

    indicator_report.append(
        {
            "indicateur": column,
            "present_train": train_exists,
            "present_test": test_exists,
            "valeurs_binaires_valides":
                binary_values_valid,
            "statut":
                "OK" if valid else "À vérifier",
        }
    )


indicator_report_df = pd.DataFrame(
    indicator_report
)


# ---------------------------------------------------------------------
# 6. Rapport général de validation
# ---------------------------------------------------------------------

validation_report_df = pd.DataFrame(
    [
        {
            "controle": "Même nombre de variables Train / Test",
            "statut": "OK" if same_number_columns else "ÉCHEC",
        },
        {
            "controle": "Mêmes noms de colonnes Train / Test",
            "statut": "OK" if same_column_names else "ÉCHEC",
        },
        {
            "controle": "Même ordre des colonnes Train / Test",
            "statut": "OK" if same_column_order else "ÉCHEC",
        },
        {
            "controle": "Aucune valeur manquante",
            "statut": "OK" if no_missing_values else "ÉCHEC",
        },
        {
            "controle": "Variables catégorielles originales supprimées",
            "statut": "OK" if categorical_encoding_valid else "ÉCHEC",
        },
        {
            "controle": "Toutes les variables finales sont numériques",
            "statut": "OK" if all_columns_numeric else "ÉCHEC",
        },
        {
            "controle": "Indicateurs binaires conformes",
            "statut": "OK" if indicators_valid else "ÉCHEC",
        },
    ]
)


# ---------------------------------------------------------------------
# 7. Affichage contextualisé
# ---------------------------------------------------------------------

print(
    "=== Validation finale du preprocessing ==="
)

print(
    "\nDimensions finales :"
)

print(
    f"X_train_processed : "
    f"{X_train_processed.shape[0]:,} observations × "
    f"{X_train_processed.shape[1]} variables"
)

print(
    f"X_test_processed  : "
    f"{X_test_processed.shape[0]:,} observations × "
    f"{X_test_processed.shape[1]} variables"
)

print(
    f"\nNombre total de NaN dans Train : "
    f"{train_total_nan:,}"
)

print(
    f"Nombre total de NaN dans Test  : "
    f"{test_total_nan:,}"
)


print(
    "\n=== Contrôles généraux ==="
)

display(
    validation_report_df
)


print(
    "\n=== Contrôle des indicateurs binaires ==="
)

display(
    indicator_report_df
)


# ---------------------------------------------------------------------
# 8. Validation définitive
# ---------------------------------------------------------------------

all_checks_valid = (
    validation_report_df["statut"]
    .eq("OK")
    .all()
)

if not all_checks_valid:

    raise ValueError(
        "La validation finale du preprocessing a échoué."
    )


print(
    "\n✅ Validation finale réussie : "
    "X_train_processed et X_test_processed "
    "sont conformes et prêts pour la modélisation."
)

=== Validation finale du preprocessing ===

Dimensions finales :
X_train_processed : 80,000 observations × 31 variables
X_test_processed  : 20,000 observations × 31 variables

Nombre total de NaN dans Train : 0
Nombre total de NaN dans Test  : 0

=== Contrôles généraux ===


,controle,statut
0,Même nombre de variables Train / Test,OK
1,Mêmes noms de colonnes Train / Test,OK
2,Même ordre des colonnes Train / Test,OK
3,Aucune valeur manquante,OK
4,Variables catégorielles originales supprimées,OK
5,Toutes les variables finales sont numériques,OK
6,Indicateurs binaires conformes,OK



=== Contrôle des indicateurs binaires ===


,indicateur,present_train,present_test,valeurs_binaires_valides,statut
0,has_electric_energy_consumption_wh_km,True,True,True,OK
1,has_electric_range_km,True,True,True,OK
2,has_fuel_consumption,True,True,True,OK
3,has_co2_reduction_wltp_g_km,True,True,True,OK



✅ Validation finale réussie : X_train_processed et X_test_processed sont conformes et prêts pour la modélisation.


### 10.16 Sauvegarde des jeux de données prétraités

Le preprocessing ayant été entièrement appliqué et validé, les jeux de données
obtenus sont maintenant sauvegardés afin d'être réutilisés dans les étapes
suivantes du projet, notamment pour l'entraînement et l'évaluation des modèles.

Les quatre jeux issus de la séparation Train / Test sont sauvegardés :

- `X_train_processed` : variables explicatives d'entraînement prétraitées ;
- `X_test_processed` : variables explicatives de test prétraitées ;
- `y_train` : cible associée au jeu d'entraînement ;
- `y_test` : cible associée au jeu de test.

Les fichiers sont enregistrés dans le répertoire prévu pour les données
prétraitées du projet.

Cette sauvegarde permet de séparer clairement la responsabilité du présent
notebook — préparation des données pour le Machine Learning — de celle des
prochaines étapes consacrées à la modélisation.

Les dimensions et la structure des données sauvegardées correspondent aux
objets validés au point 10.15.

In [26]:
# ---------------------------------------------------------------------
# 10.16 - Sauvegarde des jeux de données prétraités
# ---------------------------------------------------------------------

from pathlib import Path


# ---------------------------------------------------------------------
# 1. Détermination de la racine du projet
# ---------------------------------------------------------------------

project_root = Path.cwd().resolve()

while (
    project_root.parent != project_root
    and not (project_root / "pyproject.toml").exists()
):
    project_root = project_root.parent

if not (project_root / "pyproject.toml").exists():
    raise FileNotFoundError(
        "Impossible de déterminer la racine du projet."
    )


# ---------------------------------------------------------------------
# 2. Répertoire de sauvegarde
# ---------------------------------------------------------------------

processed_data_dir = (
    project_root
    / "data"
    / "processed"
)

processed_data_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------------------
# 3. Fichiers de sortie
# ---------------------------------------------------------------------

x_train_path = (
    processed_data_dir
    / "X_train_processed.parquet"
)

x_test_path = (
    processed_data_dir
    / "X_test_processed.parquet"
)

y_train_path = (
    processed_data_dir
    / "y_train.parquet"
)

y_test_path = (
    processed_data_dir
    / "y_test.parquet"
)


# ---------------------------------------------------------------------
# 4. Sauvegarde
# ---------------------------------------------------------------------

X_train_processed.to_parquet(
    x_train_path,
    index=True,
)

X_test_processed.to_parquet(
    x_test_path,
    index=True,
)

y_train.to_frame().to_parquet(
    y_train_path,
    index=True,
)

y_test.to_frame().to_parquet(
    y_test_path,
    index=True,
)


# ---------------------------------------------------------------------
# 5. Vérification de la sauvegarde
# ---------------------------------------------------------------------

saved_files = {
    "X_train_processed": x_train_path,
    "X_test_processed": x_test_path,
    "y_train": y_train_path,
    "y_test": y_test_path,
}

missing_saved_files = [
    name
    for name, path in saved_files.items()
    if not path.exists()
]

if missing_saved_files:
    raise FileNotFoundError(
        "Échec de sauvegarde pour : "
        + ", ".join(missing_saved_files)
    )


# ---------------------------------------------------------------------
# 6. Rapport métier de sauvegarde
# ---------------------------------------------------------------------

saved_datasets_report_df = pd.DataFrame(
    [
        {
            "dataset": "X_train_processed",
            "role": "Variables explicatives - entraînement",
            "observations": X_train_processed.shape[0],
            "variables": X_train_processed.shape[1],
            "statut": "Sauvegardé",
        },
        {
            "dataset": "X_test_processed",
            "role": "Variables explicatives - test",
            "observations": X_test_processed.shape[0],
            "variables": X_test_processed.shape[1],
            "statut": "Sauvegardé",
        },
        {
            "dataset": "y_train",
            "role": "Variable cible - entraînement",
            "observations": len(y_train),
            "variables": 1,
            "statut": "Sauvegardé",
        },
        {
            "dataset": "y_test",
            "role": "Variable cible - test",
            "observations": len(y_test),
            "variables": 1,
            "statut": "Sauvegardé",
        },
    ]
)

display(saved_datasets_report_df)


print(
    "\n✅ Les quatre jeux de données prétraités "
    "ont été sauvegardés dans data/processed/."
)

,dataset,role,observations,variables,statut
0,X_train_processed,Variables explicatives - entraînement,80000,31,Sauvegardé
1,X_test_processed,Variables explicatives - test,20000,31,Sauvegardé
2,y_train,Variable cible - entraînement,80000,1,Sauvegardé
3,y_test,Variable cible - test,20000,1,Sauvegardé



✅ Les quatre jeux de données prétraités ont été sauvegardés dans data/processed/.


### 10.17 Sauvegarde des artefacts de preprocessing

Les jeux de données prétraités ont été sauvegardés au point précédent.

Afin de rendre le preprocessing reproductible sur de nouvelles données,
les objets et paramètres appris exclusivement à partir des données
d'entraînement doivent également être persistés.

Les artefacts sauvegardés sont :

- les valeurs d'imputation apprises sur `X_train` ;
- le `OneHotEncoder` appris sur les trois variables catégorielles ;
- le `StandardScaler` appris sur les variables numériques continues ;
- les métadonnées décrivant la structure finale du preprocessing.

Les valeurs d'imputation permettent de réutiliser exactement, lors de
l'inférence, les médianes apprises pendant l'entraînement sans recalculer
de statistiques sur de nouvelles données.

Les métadonnées permettent notamment de conserver :

- les variables traitées par One-Hot Encoding ;
- les variables générées par One-Hot Encoding ;
- les indicateurs binaires créés ;
- les variables standardisées ;
- l'ordre exact des variables finales utilisées pour la modélisation.

Ces artefacts sont enregistrés dans le répertoire `models/preprocessing/`.

In [27]:
# ---------------------------------------------------------------------
# 10.17 - Sauvegarde des artefacts de preprocessing
# ---------------------------------------------------------------------

import json
import joblib


# ---------------------------------------------------------------------
# 1. Vérification de la racine du projet
# ---------------------------------------------------------------------

if "project_root" not in globals():

    project_root = Path.cwd().resolve()

    while (
        project_root.parent != project_root
        and not (project_root / "pyproject.toml").exists()
    ):
        project_root = project_root.parent

    if not (project_root / "pyproject.toml").exists():
        raise FileNotFoundError(
            "Impossible de déterminer la racine du projet."
        )


# ---------------------------------------------------------------------
# 2. Répertoire de sauvegarde des artefacts
# ---------------------------------------------------------------------

preprocessing_dir = (
    project_root
    / "models"
    / "preprocessing"
)

preprocessing_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------------------
# 3. Définition des fichiers de sortie
# ---------------------------------------------------------------------

imputation_values_path = (
    preprocessing_dir
    / "imputation_values.joblib"
)

onehot_encoder_path = (
    preprocessing_dir
    / "onehot_encoder.joblib"
)

standard_scaler_path = (
    preprocessing_dir
    / "standard_scaler.joblib"
)

metadata_path = (
    preprocessing_dir
    / "preprocessing_metadata.json"
)


# ---------------------------------------------------------------------
# 4. Construction des valeurs d'imputation apprises sur Train
# ---------------------------------------------------------------------

imputation_values = {}


# Valeurs issues du traitement conditionnel du point 10.6
for _, row in conditional_imputation_report_df.iterrows():

    imputation_values[
        row["variable"]
    ] = float(
        row["median_train"]
    )


# Valeurs issues de l'imputation des NaN rares du point 10.8
for variable in rare_missing_columns:

    variable_rows = (
        rare_imputation_report_df[
            rare_imputation_report_df["variable"] == variable
        ]
    )

    if variable_rows.empty:
        raise ValueError(
            f"Aucune valeur d'imputation trouvée pour '{variable}'."
        )

    imputation_values[
        variable
    ] = float(
        variable_rows.iloc[0]["median_train"]
    )


# ---------------------------------------------------------------------
# 5. Sauvegarde des objets appris sur Train
# ---------------------------------------------------------------------

joblib.dump(
    imputation_values,
    imputation_values_path,
)

joblib.dump(
    onehot_encoder,
    onehot_encoder_path,
)

joblib.dump(
    standard_scaler,
    standard_scaler_path,
)


# ---------------------------------------------------------------------
# 6. Construction des métadonnées du preprocessing
# ---------------------------------------------------------------------

preprocessing_metadata = {

    "imputation": {
        "artifact": "imputation_values.joblib",
        "columns": list(
            imputation_values.keys()
        ),
    },

    "onehot_encoding": {
        "source_columns": list(
            onehot_columns
        ),
        "output_columns": list(
            onehot_feature_names
        ),
        "handle_unknown": "ignore",
    },

    "binary_indicators": list(
        binary_indicator_columns
    ),

    "standardization": {
        "columns": list(
            columns_to_scale
        ),
    },

    "final_features": (
        X_train_processed
        .columns
        .tolist()
    ),
}


# ---------------------------------------------------------------------
# 7. Sauvegarde des métadonnées
# ---------------------------------------------------------------------

with open(
    metadata_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        preprocessing_metadata,
        file,
        indent=4,
        ensure_ascii=False,
    )


# ---------------------------------------------------------------------
# 8. Vérification de la sauvegarde
# ---------------------------------------------------------------------

saved_artifacts = {

    "Valeurs d'imputation":
        imputation_values_path,

    "OneHotEncoder":
        onehot_encoder_path,

    "StandardScaler":
        standard_scaler_path,

    "Métadonnées preprocessing":
        metadata_path,
}

missing_artifacts = [
    name
    for name, path in saved_artifacts.items()
    if not path.exists()
]

if missing_artifacts:
    raise FileNotFoundError(
        "Échec de sauvegarde pour : "
        + ", ".join(missing_artifacts)
    )


# ---------------------------------------------------------------------
# 9. Rapport fonctionnel
# ---------------------------------------------------------------------

artifacts_report_df = pd.DataFrame(
    [
        {
            "artefact": "Valeurs d'imputation",
            "role": (
                "Médianes apprises exclusivement sur Train"
            ),
            "statut": "Sauvegardé",
        },
        {
            "artefact": "OneHotEncoder",
            "role": (
                "Encodage des variables catégorielles"
            ),
            "statut": "Sauvegardé",
        },
        {
            "artefact": "StandardScaler",
            "role": (
                "Standardisation des variables numériques"
            ),
            "statut": "Sauvegardé",
        },
        {
            "artefact": "Métadonnées preprocessing",
            "role": (
                "Structure et paramètres du preprocessing"
            ),
            "statut": "Sauvegardé",
        },
    ]
)


# ---------------------------------------------------------------------
# 10. Résumé contextualisé
# ---------------------------------------------------------------------

print(
    "=== Sauvegarde des artefacts de preprocessing ==="
)

print(
    f"\nRépertoire de sauvegarde : "
    f"{preprocessing_dir}"
)

print(
    f"\nNombre d'artefacts sauvegardés : "
    f"{len(saved_artifacts)}"
)

display(
    artifacts_report_df
)

print(
    "\nValeurs d'imputation sauvegardées :"
)

for column, value in imputation_values.items():
    print(
        f"  - {column} : {value}"
    )

print(
    "\nNombre de variables finales enregistrées "
    f"dans les métadonnées : "
    f"{len(preprocessing_metadata['final_features'])}"
)

print(
    "\n✅ Les artefacts de preprocessing ont été "
    "sauvegardés dans models/preprocessing/."
)

=== Sauvegarde des artefacts de preprocessing ===

Répertoire de sauvegarde : /home/jmbandong/projects/ml-projects/vehicle-emissions-prediction-mlops/models/preprocessing

Nombre d'artefacts sauvegardés : 4


,artefact,role,statut
0,Valeurs d'imputation,Médianes apprises exclusivement sur Train,Sauvegardé
1,OneHotEncoder,Encodage des variables catégorielles,Sauvegardé
2,StandardScaler,Standardisation des variables numériques,Sauvegardé
3,Métadonnées preprocessing,Structure et paramètres du preprocessing,Sauvegardé



Valeurs d'imputation sauvegardées :
  - engine_capacity_cm3 : 1498.0
  - electric_energy_consumption_wh_km : 167.0
  - electric_range_km : 394.0
  - fuel_consumption : 5.5
  - wltp_test_mass_kg : 1609.0
  - engine_power_kw : 103.0

Nombre de variables finales enregistrées dans les métadonnées : 31

✅ Les artefacts de preprocessing ont été sauvegardés dans models/preprocessing/.


### 10.18 Validation du rechargement des artefacts de preprocessing

Les artefacts de preprocessing ayant été sauvegardés, leur rechargement est
contrôlé avant de clôturer le notebook.

Cette étape vérifie que :

- les valeurs d'imputation apprises sur Train peuvent être rechargées ;
- le `OneHotEncoder` peut être rechargé ;
- le `StandardScaler` peut être rechargé ;
- les métadonnées JSON peuvent être relues ;
- les catégories apprises par le `OneHotEncoder` sont disponibles ;
- les paramètres appris par le `StandardScaler` sont disponibles ;
- les valeurs d'imputation rechargées correspondent aux variables enregistrées
  dans les métadonnées ;
- le nombre et l'ordre des variables finales enregistrées dans les métadonnées
  correspondent exactement à la structure de `X_train_processed` et
  `X_test_processed`.

Aucun nouvel apprentissage ni aucune transformation des données n'est effectué
à cette étape.

L'objectif est de vérifier que tous les artefacts persistés sont effectivement
réutilisables pour reproduire le preprocessing sur de nouvelles données.

In [28]:
# ---------------------------------------------------------------------
# 10.18 - Validation du rechargement des artefacts de preprocessing
# ---------------------------------------------------------------------


# ---------------------------------------------------------------------
# 1. Rechargement des artefacts sauvegardés
# ---------------------------------------------------------------------

loaded_imputation_values = joblib.load(
    imputation_values_path
)

loaded_onehot_encoder = joblib.load(
    onehot_encoder_path
)

loaded_standard_scaler = joblib.load(
    standard_scaler_path
)

with open(
    metadata_path,
    "r",
    encoding="utf-8",
) as file:

    loaded_metadata = json.load(file)


# ---------------------------------------------------------------------
# 2. Récupération des informations enregistrées
# ---------------------------------------------------------------------

saved_final_features = loaded_metadata[
    "final_features"
]

saved_imputation_columns = loaded_metadata[
    "imputation"
]["columns"]

saved_onehot_columns = loaded_metadata[
    "onehot_encoding"
]["source_columns"]

saved_onehot_output_columns = loaded_metadata[
    "onehot_encoding"
]["output_columns"]

saved_scaling_columns = loaded_metadata[
    "standardization"
]["columns"]

saved_binary_indicators = loaded_metadata[
    "binary_indicators"
]


# ---------------------------------------------------------------------
# 3. Structure actuelle de Train et Test
# ---------------------------------------------------------------------

train_final_features = (
    X_train_processed
    .columns
    .tolist()
)

test_final_features = (
    X_test_processed
    .columns
    .tolist()
)

metadata_matches_train = (
    saved_final_features
    == train_final_features
)

metadata_matches_test = (
    saved_final_features
    == test_final_features
)

train_matches_test = (
    train_final_features
    == test_final_features
)


# ---------------------------------------------------------------------
# 4. Validation des artefacts rechargés
# ---------------------------------------------------------------------

imputation_values_valid = (
    isinstance(
        loaded_imputation_values,
        dict,
    )
    and set(
        loaded_imputation_values.keys()
    )
    == set(
        saved_imputation_columns
    )
    and all(
        pd.notna(value)
        for value in loaded_imputation_values.values()
    )
)

onehot_encoder_valid = (
    hasattr(
        loaded_onehot_encoder,
        "categories_",
    )
    and len(
        loaded_onehot_encoder.categories_
    )
    == len(saved_onehot_columns)
)

standard_scaler_valid = (
    hasattr(
        loaded_standard_scaler,
        "mean_",
    )
    and hasattr(
        loaded_standard_scaler,
        "scale_",
    )
    and len(
        loaded_standard_scaler.mean_
    )
    == len(saved_scaling_columns)
    and len(
        loaded_standard_scaler.scale_
    )
    == len(saved_scaling_columns)
)

metadata_structure_valid = (
    isinstance(saved_final_features, list)
    and isinstance(saved_imputation_columns, list)
    and isinstance(saved_onehot_columns, list)
    and isinstance(saved_onehot_output_columns, list)
    and isinstance(saved_scaling_columns, list)
    and isinstance(saved_binary_indicators, list)
)


# ---------------------------------------------------------------------
# 5. Rapport général de validation
# ---------------------------------------------------------------------

artifact_validation_df = pd.DataFrame(
    [
        {
            "artefact": "Valeurs d'imputation",
            "verification": (
                "Médianes apprises sur Train rechargeables"
            ),
            "statut": (
                "OK"
                if imputation_values_valid
                else "À vérifier"
            ),
        },
        {
            "artefact": "OneHotEncoder",
            "verification": (
                "Encodeur et catégories apprises rechargeables"
            ),
            "statut": (
                "OK"
                if onehot_encoder_valid
                else "À vérifier"
            ),
        },
        {
            "artefact": "StandardScaler",
            "verification": (
                "Moyennes et écarts-types appris rechargeables"
            ),
            "statut": (
                "OK"
                if standard_scaler_valid
                else "À vérifier"
            ),
        },
        {
            "artefact": "Métadonnées",
            "verification": (
                "Structure JSON conforme"
            ),
            "statut": (
                "OK"
                if metadata_structure_valid
                else "À vérifier"
            ),
        },
        {
            "artefact": "Métadonnées",
            "verification": (
                "Variables finales identiques à X_train_processed"
            ),
            "statut": (
                "OK"
                if metadata_matches_train
                else "À vérifier"
            ),
        },
        {
            "artefact": "Métadonnées",
            "verification": (
                "Variables finales identiques à X_test_processed"
            ),
            "statut": (
                "OK"
                if metadata_matches_test
                else "À vérifier"
            ),
        },
        {
            "artefact": "Train / Test",
            "verification": (
                "Même nombre, mêmes noms et même ordre "
                "des variables"
            ),
            "statut": (
                "OK"
                if train_matches_test
                else "À vérifier"
            ),
        },
    ]
)


# ---------------------------------------------------------------------
# 6. Résumé général
# ---------------------------------------------------------------------

print(
    "=== Validation du rechargement "
    "des artefacts de preprocessing ==="
)

print(
    "\nLes artefacts sauvegardés sont rechargés "
    "sans nouvel apprentissage."
)

display(
    artifact_validation_df
)


# ---------------------------------------------------------------------
# 7. Détail des valeurs d'imputation
# ---------------------------------------------------------------------

print("\n" + "=" * 70)
print("1. VALEURS D'IMPUTATION")
print("=" * 70)

imputation_details_df = pd.DataFrame(
    [
        {
            "variable": column,
            "valeur_imputation_train": value,
        }
        for column, value
        in loaded_imputation_values.items()
    ]
)

display(
    imputation_details_df
)

print(
    f"\nNombre de variables avec une valeur "
    f"d'imputation sauvegardée : "
    f"{len(loaded_imputation_values)}"
)


# ---------------------------------------------------------------------
# 8. Détail du One-Hot Encoding
# ---------------------------------------------------------------------

print("\n" + "=" * 70)
print("2. ONE-HOT ENCODING")
print("=" * 70)

onehot_details = []

for column, categories in zip(
    saved_onehot_columns,
    loaded_onehot_encoder.categories_,
):

    onehot_details.append(
        {
            "variable_source": column,
            "nombre_modalites_apprises": len(categories),
            "modalites_apprises": ", ".join(
                map(str, categories)
            ),
        }
    )

onehot_details_df = pd.DataFrame(
    onehot_details
)

display(
    onehot_details_df
)

print(
    f"\nNombre total de variables binaires créées "
    f"par One-Hot Encoding : "
    f"{len(saved_onehot_output_columns)}"
)


# ---------------------------------------------------------------------
# 9. Détail du StandardScaler
# ---------------------------------------------------------------------

print("\n" + "=" * 70)
print("3. STANDARDISATION")
print("=" * 70)

scaler_details_df = pd.DataFrame(
    {
        "variable": saved_scaling_columns,
        "moyenne_apprise_train":
            loaded_standard_scaler.mean_,
        "ecart_type_appris_train":
            loaded_standard_scaler.scale_,
    }
)

display(
    scaler_details_df
)

print(
    f"\nNombre de variables standardisées : "
    f"{len(saved_scaling_columns)}"
)


# ---------------------------------------------------------------------
# 10. Détail des indicateurs binaires
# ---------------------------------------------------------------------

print("\n" + "=" * 70)
print("4. INDICATEURS BINAIRES CONSERVÉS")
print("=" * 70)

binary_indicators_df = pd.DataFrame(
    {
        "indicateur": saved_binary_indicators,
        "traitement": (
            ["Conservé en 0/1 sans standardisation"]
            * len(saved_binary_indicators)
        ),
    }
)

display(
    binary_indicators_df
)


# ---------------------------------------------------------------------
# 11. Structure finale utilisée pour la modélisation
# ---------------------------------------------------------------------

print("\n" + "=" * 70)
print("5. STRUCTURE FINALE")
print("=" * 70)

print(
    f"\nNombre total de variables finales : "
    f"{len(saved_final_features)}"
)

final_features_df = pd.DataFrame(
    {
        "position": range(
            1,
            len(saved_final_features) + 1,
        ),
        "variable_finale": saved_final_features,
    }
)

display(
    final_features_df
)


# ---------------------------------------------------------------------
# 12. Validation stricte
# ---------------------------------------------------------------------

all_artifacts_valid = all(
    [
        imputation_values_valid,
        onehot_encoder_valid,
        standard_scaler_valid,
        metadata_structure_valid,
        metadata_matches_train,
        metadata_matches_test,
        train_matches_test,
    ]
)

if not all_artifacts_valid:

    raise ValueError(
        "La validation des artefacts "
        "de preprocessing a échoué."
    )


print(
    "\n✅ Validation réussie : tous les artefacts sauvegardés "
    "sont rechargeables et correspondent à la structure "
    "finale utilisée pour la modélisation."
)

=== Validation du rechargement des artefacts de preprocessing ===

Les artefacts sauvegardés sont rechargés sans nouvel apprentissage.


,artefact,verification,statut
0,Valeurs d'imputation,Médianes apprises sur Train rechargeables,OK
1,OneHotEncoder,Encodeur et catégories apprises rechargeables,OK
2,StandardScaler,Moyennes et écarts-types appris rechargeables,OK
3,Métadonnées,Structure JSON conforme,OK
4,Métadonnées,Variables finales identiques à X_train_processed,OK
5,Métadonnées,Variables finales identiques à X_test_processed,OK
6,Train / Test,"Même nombre, mêmes noms et même ordre des vari...",OK



1. VALEURS D'IMPUTATION


,variable,valeur_imputation_train
0,engine_capacity_cm3,1498.0
1,electric_energy_consumption_wh_km,167.0
2,electric_range_km,394.0
3,fuel_consumption,5.5
4,wltp_test_mass_kg,1609.0
5,engine_power_kw,103.0



Nombre de variables avec une valeur d'imputation sauvegardée : 6

2. ONE-HOT ENCODING


...


Nombre de variables standardisées : 10

4. INDICATEURS BINAIRES CONSERVÉS


,indicateur,traitement
0,has_electric_energy_consumption_wh_km,Conservé en 0/1 sans standardisation
1,has_electric_range_km,Conservé en 0/1 sans standardisation
2,has_fuel_consumption,Conservé en 0/1 sans standardisation
3,has_co2_reduction_wltp_g_km,Conservé en 0/1 sans standardisation



5. STRUCTURE FINALE

Nombre total de variables finales : 31


,position,variable_finale
0,1,mass_running_order_kg
1,2,wltp_test_mass_kg
2,3,engine_capacity_cm3
3,4,engine_power_kw
4,5,electric_energy_consumption_wh_km
5,6,co2_reduction_wltp_g_km
6,7,fuel_consumption
7,8,electric_range_km
8,9,registration_month_sin
9,10,registration_month_cos



✅ Validation réussie : tous les artefacts sauvegardés sont rechargeables et correspondent à la structure finale utilisée pour la modélisation.


### 10.19 Conclusion du preprocessing Train / Test

Le preprocessing des données d'entraînement et de test est désormais terminé,
sauvegardé et validé.

Les traitements réalisés dans ce notebook ont permis de construire des jeux de
données directement exploitables pour la phase de modélisation, tout en
respectant strictement la séparation entre Train et Test afin d'éviter les
fuites d'information.

Les principales étapes réalisées sont :

- séparation des variables explicatives et de la variable cible ;
- exclusion de `manufacturer_make` des variables utilisées pour la modélisation ;
- constitution de `X_train`, `X_test`, `y_train` et `y_test` ;
- analyse des valeurs manquantes et distinction entre absences structurelles,
  résiduelles rares et absence informative ;
- création de quatre indicateurs binaires permettant de conserver l'information
  associée à certaines valeurs initialement manquantes ;
- application de règles métier pour le traitement des valeurs manquantes
  structurelles ;
- imputation des valeurs manquantes résiduelles à partir de médianes apprises
  exclusivement sur Train ;
- traitement spécifique de `co2_reduction_wltp_g_km`, dont les valeurs
  manquantes sont remplacées par `0` tout en conservant leur absence initiale
  grâce à l'indicateur associé ;
- One-Hot Encoding de `vehicle_category_type`, `fuel_type` et `fuel_mode`,
  appris exclusivement sur Train ;
- standardisation des variables numériques continues à partir des paramètres
  appris exclusivement sur Train ;
- conservation sans standardisation des quatre indicateurs binaires et des
  variables issues du One-Hot Encoding ;
- validation de l'absence de valeurs manquantes et de variables non numériques ;
- vérification de la cohérence des structures finales entre Train et Test ;
- sauvegarde des jeux de données prétraités au format Parquet ;
- sauvegarde des valeurs d'imputation apprises sur Train, du `OneHotEncoder`,
  du `StandardScaler` et des métadonnées de preprocessing ;
- rechargement et validation des artefacts sauvegardés.

Les matrices finales obtenues sont :

- `X_train_processed` : **80 000 observations × 31 variables** ;
- `X_test_processed` : **20 000 observations × 31 variables**.

Les deux matrices possèdent exactement les mêmes variables, dans le même ordre,
et ne contiennent aucune valeur manquante.

La structure finale comprend :

- **10 variables numériques continues standardisées** ;
- **4 indicateurs binaires** conservant l'information sur certaines valeurs
  initialement manquantes ;
- **17 variables binaires** issues du One-Hot Encoding.

Les artefacts nécessaires à la reproduction du preprocessing ont également été
persistés :

- `imputation_values.joblib` ;
- `onehot_encoder.joblib` ;
- `standard_scaler.joblib` ;
- `preprocessing_metadata.json`.

Les métadonnées enregistrent notamment les variables utilisées par les
transformations ainsi que l'ordre exact des **31 variables finales**.

Le preprocessing effectué dans ce notebook est ainsi reproductible sans
réapprendre les paramètres à partir des données de test ou des futures données
d'inférence.

Les données prétraitées sont maintenant prêtes pour l'étape suivante du projet :
**l'entraînement, la comparaison et l'évaluation des modèles de Machine
Learning**.